# Non Land Water Use Analysis 🌆🌳
Database provided by CR2 on 24 April 2023

Mariana Bruning-González mariana.bruning@alumnos.uach.cl Doctorado en Ecología y Evolución, Universidad Austral de Chile

Packages
==================

Install packages
------------------------------

In [ ]:
! pip install pandas
! pip install geopandas
! pip install plotly.express
! pip install matplotlib
! pip install geojson
! pip install ipykernel
#! pip install "jupyterlab>=3" "ipywidgets>=7.6"
! pip install plotly #quizás no es necesario con plotly express?
! pip install ast
! pip install nbformat 
! pip install --upgrade nbformat
! pip install geoplot
! pip install statsmodels
! pip3 install geojson 
! pip install statsmodels

In [ ]:
! pip install pyarrow
! pip install fastparquet

In [ ]:
! pip install -U kaleido

Import packages
-------------

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import geopandas as gpd 
import matplotlib.pyplot as plt
import plotly.express as px 
import plotly.graph_objs as pg
import datetime
import numpy as np
import scipy.interpolate
import statsmodels
import requests

In [ ]:
import geojson
from geojson import GeometryCollection, Point, LineString, Feature, FeatureCollection
import ast
import statsmodels.api as sm
import os
import networkx as nx
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Ingreso a carpetas, lectura y limpieza de datos / Entry folders, data lecture and cleaning

## Directorio de la base de datos / Database directory

In [ ]:
# Verifica el directorio actual / Check current path
print("Directorio actual:", os.getcwd())


In [ ]:
# Establecer ruta / Set path
#path = r'/home/mari/Documents/GitHub/tesis'
path = r'/Users/marianabruninggonzalez/Desktop/Modelostesisjulio2025/Python'

# Cambia el directorio actual / Update current path
os.chdir(path)

In [ ]:
ls

## Directorio para guardar imagenes / Saved images path

In [ ]:
imagen_path = r'/Users/marianabruninggonzalez/Desktop/Modelostesisjulio2025/Python/Imagen'


## Lectura de datos

In [ ]:
# ANTIGUOS
#Leer base de datos NLWU / Read NLWU database
df = pd.read_parquet(path+'/CR2/bbdd_cr2_01.parquet')
df2 = pd.read_parquet(path+'/CR2/bbdd_cr2_02.parquet')

In [ ]:
df.head()

In [ ]:
# NUEVOS OCTUBRE 2025
df_tot_wu_consunt = pd.read_csv(path+'/df_tot_wu_consunt.csv')
df_tot = pd.read_csv(path+'/df_tot.csv')

In [ ]:
# Esta celda, al correrla desde arriba, me tira error... debe haber sido una prueba que hice en algún momento 13/03
df[df['nom_reg']=='REGIÓN METROPOLITANA DE SANTIAGO'].groupby('Level_1','Years')['Value'].mean()
#df_prueba2 = df2.groupby('Years')['Value'].mean()

#wos_um[wos_um['Author Keywords'] == 'metabolism urban']

In [ ]:
df2.head()

In [ ]:
df2.shape

In [ ]:
# Archivo de nombres y códigos / Names and codes file
cod = pd.read_csv(path + r'/CR2/LUT_cod_com_Chile.csv')

#dictionary code region : name region
cod_reg_nom_reg = cod[['cod_reg', 'nom_reg']].drop_duplicates().set_index('cod_reg')['nom_reg'].astype(str).to_dict()
cod_reg_nom_reg = {str(k): v for k, v in cod_reg_nom_reg.items()} #same type of data as csv column names NLU_XX

#dictionary code commune : name commune
cod_com_nom_com = cod[['cod_com', 'nom_com']].drop_duplicates().set_index('cod_com')['nom_com'].astype(str).to_dict()
cod_com_nom_com = {str(k): v for k, v in cod_com_nom_com.items()} 

## Diccionarios útiles / Usefull dictionaries

In [ ]:
# Sectores para plotear en inglés / Levels to plot in english
english_level = {
    'DrinkingWater': 'Drinking Water',
}

In [ ]:
# Sectores de inglés a español / Levels from English to Spanish 
english_to_spanish = {
    'Drinking Water': 'Agua Potable',
    'Energy': 'Energía',
    'Industry': 'Industria',
    'Livestock': 'Pecuario',
    'Mining': 'Minería',
}

In [ ]:
# Macrozonas en inglés  / Macrozones in english
spanish_to_english = {
    'Macrozona Norte': 'Northern Macrozone',
    'Macrozona Centro': 'Central Macrozone',
    'Macrozona Centro Sur': 'Central-Southern Macrozone',
    'Macrozona Sur': 'Southern Macrozone',
    'Macrozona Austral': 'Austral Macrozone'
}

In [ ]:
# Colores de los sectores / Level's colors
colores_sector = {
    'Drinking Water': '#377eb8',
    'Energy': '#dede00',
    'Industry': '#999999',
    'Livestock': '#f781bf',
    'Mining': '#ff7f00',
    'Agua Potable': '#377eb8',
    'Energía': '#dede00',
    'Industria': '#999999',
    'Pecuario': '#f781bf',
    'Minería': '#ff7f00'  
}

# Reorganizar dataframe /Dataframe reorganization

In [ ]:
# Solamente uso consuntivo / Consumptive use only
df2_filtered = df2.loc[df2['Level_2'] != 'Hydraulic']

In [ ]:
# Toda la base de datos 

# Primero, crea una tabla pivote con los valores de consumo hídrico para cada sector
pivot_table = df.pivot_table(index=['Years', 'cod_com', 'nom_com', 'cod_reg', 'nom_reg'],
                            columns='Level_1',
                             values='Value',
                             aggfunc='sum')

# Reinicia el índice para que los niveles de índice se conviertan en columnas
pivot_table.reset_index(inplace=True)

# Si hay valores faltantes (NaN), llénalos con cero
pivot_table.fillna(0, inplace=True)

# Visualiza el nuevo DataFrame
print(pivot_table.head())



In [ ]:
# Solamente uso consuntivo

# Primero, crea una tabla pivote con los valores de consumo hídrico para cada sector
pivot_table_f = df2_filtered.pivot_table(index=['Years', 'cod_com', 'nom_com', 'cod_reg', 'nom_reg'],
                            columns='Level_1',
                             values='Value',
                             aggfunc='sum')

# Reinicia el índice para que los niveles de índice se conviertan en columnas
pivot_table_f.reset_index(inplace=True)

# Si hay valores faltantes (NaN), llénalos con cero
pivot_table_f.fillna(0, inplace=True)

In [ ]:
# Verificar de ambas tablas no tengan la misma información

print(pivot_table['Energy'].sum())
print(pivot_table_f['Energy'].sum())

### Verificación pivoteo


In [ ]:
# Dataframe original

# Filtrar el DataFrame original para la región con código 1 y el sector DrinkingWater
consumo_reg_1_drinking_water = df[(df['cod_reg'] == 1) & (df['Level_1'] == 'DrinkingWater')]
# Sumar el consumo hídrico para el sector DrinkingWater
total_consumo_reg_1_drinking_water = consumo_reg_1_drinking_water['Value'].sum()
print("Consumo hídrico total para la región con código 1 y el sector DrinkingWater:", total_consumo_reg_1_drinking_water)



# Dataframe pivoteado

# Filtrar el nuevo DataFrame pivoteado para la región con código 1
consumo_reg_1_pivot = pivot_table[pivot_table['cod_reg'] == 1]
# Obtener el consumo de agua para la columna DrinkingWater
consumo_reg_1_drinking_water_pivot = consumo_reg_1_pivot['DrinkingWater']
print("Consumo hídrico total para la región con código 1 y el sector DrinkingWater (desde el DataFrame pivoteado):", consumo_reg_1_drinking_water_pivot.sum())


# Gráficos generales / General plots

## Gráficos generales nuevo

# revisando por qué no me da lo mismo que antes (10/2025 vs meses atrás)
en particular, el gráfico nacional de 

In [ ]:

print(by_year_nlwu['Energy'].sum())
print(by_year['Energy'].sum())
print(df_tot['Energy'].sum()) # entonces df tot tiene el uso consuntivo y no consuntivo
print(df_tot_wu_consunt['Energy'].sum())

In [ ]:
print(df_tot_wu_consunt['Irrigated'].sum())
df_tot_wu_consunt['NLWU'] = df_tot_wu_consunt['DrinkingWater']+df_tot_wu_consunt['Energy']+df_tot_wu_consunt['Mining']+df_tot_wu_consunt['Livestock']+df_tot_wu_consunt['Industry']
print(df_tot_wu_consunt['NLWU'].sum())

In [ ]:
print(df_tot_wu_consunt[df_tot_wu_consunt['Years']==2020]['Irrigated'].sum())
print(df_tot_wu_consunt[df_tot_wu_consunt['Years']==2020]['NLWU'].sum())

In [ ]:
# Diccionario de colores 
sector_colors = {
    "DrinkingWater": "#0072b2",
    "Energy": "#f0e442",
    "Industry": "#cc79a7",
    "Mining": "#e69f00",
    "Livestock": "#d55e00",
    "Irrigated": "#009e73",
    "NLWU":"#56B4E9",    
}


In [ ]:
# WU
# 1) (opcional pero útil) limpiar nombres de columnas
df = df_tot_wu_consunt.copy()
df = df[df['Years'] >= 1960]
df.columns = (
    df.columns
      .str.strip()
      .str.replace(r'\s+', '_', regex=True)
      .str.replace(r'[^\w]+', '_', regex=True)
)

# 2) define las columnas de sectores (ajusta si tus nombres difieren)
#sectors = ['DrinkingWater','Energy','Industry','Mining','Livestock','Irrigated']
sectors = ['Livestock','Industry','Energy','Mining','DrinkingWater','Irrigated']


# 3) totales por año (nivel país)
by_year = (
    df.groupby('Years', as_index=False)[sectors]
      .sum(min_count=1)   # respeta NaN si un año está vacío en todos los sectores
)

# 4) total nacional por año
by_year['Total'] = by_year[sectors].sum(axis=1)

# 5) porcentajes por año (para barras apiladas que suman 100)
pct = by_year[sectors].div(by_year['Total'], axis=0) * 100
pct.columns = [c + '_pct' for c in pct.columns]

# 6) juntar todo en un solo DataFrame
by_year = pd.concat([by_year, pct], axis=1)

by_year.head()


In [ ]:
# NLWU
# 1) (opcional pero útil) limpiar nombres de columnas
df_NLWU = df.copy()
df_NLWU = df_NLWU.drop(columns=['Irrigated'])

# 2) define las columnas de sectores (ajusta si tus nombres difieren)
sector_nlwu = ['Livestock','Industry','Energy','Mining','DrinkingWater']


# 3) totales por año (nivel país)
by_year_nlwu = (
    df_NLWU.groupby('Years', as_index=False)[sector_nlwu]
      .sum(min_count=1)   # respeta NaN si un año está vacío en todos los sectores
)

# 4) total nacional por año
by_year_nlwu['Total'] = by_year_nlwu[sector_nlwu].sum(axis=1)

# 5) porcentajes por año (para barras apiladas que suman 100)
pct_nlwu = by_year_nlwu[sector_nlwu].div(by_year_nlwu['Total'], axis=0) * 100
pct_nlwu.columns = [c + '_pct' for c in pct_nlwu.columns]

# 6) juntar todo en un solo DataFrame
by_year_nlwu = pd.concat([by_year_nlwu, pct_nlwu], axis=1)

by_year_nlwu.head()


In [ ]:
print(by_year_nlwu['Energy'].sum())
print(by_year['Energy'].sum())
#print(by_year_nlwu['Irrigated'].sum())
print(by_year['Irrigated'].sum())
print(by_year['DrinkingWater'].sum())
print(by_year_nlwu['DrinkingWater'].sum())
print(by_year_nlwu['Total'].sum())
print(by_year['Total'].sum())

In [ ]:
# WU para hacer trendline LOWESS

import numpy as np
from statsmodels.nonparametric.smoothers_lowess import lowess

# ordenamos por año y quitamos NaN en Total
#tmp = by_year[['Years','Total']].dropna().sort_values('Years')
tmp = by_year.loc[by_year['Years'] >= 1960, ['Years','Total']].sort_values('Years')
x = tmp['Years'].to_numpy()
y = tmp['Total'].to_numpy()

# LOWESS: ajusta frac (suavizado) a tu gusto: 0.2–0.4 suele ir bien
y_lowess = lowess(y, x, frac=0.1, it=1, return_sorted=False)


In [ ]:
# NLWU para hacer trendline LOWESS

# ordenamos por año y quitamos NaN en Total
tmp_nlwu = by_year_nlwu[['Years','Total']].dropna().sort_values('Years')
x_nlwu = tmp_nlwu['Years'].to_numpy()
y_nlwu = tmp_nlwu['Total'].to_numpy()

# LOWESS: ajusta frac (suavizado) a tu gusto: 0.2–0.4 suele ir bien
y_lowess_nlwu = lowess(y_nlwu, x_nlwu, frac=0.1, it=1, return_sorted=False)


In [ ]:
# WU
fig = make_subplots(specs=[[{"secondary_y": True}]])

# 1) Barras apiladas (100%) con porcentajes
for sector in sectors:
    fig.add_trace(
        go.Bar(
            x=by_year['Years'],
            y=by_year[sector + '_pct'],
            name=sector,
            marker=dict(color=sector_colors.get(sector, None))  
        ),
        secondary_y=False
    )
fig.update_traces(width=0.99,
                  marker=dict(line=dict(color='#000000',width=1))
                  )

# 2) Línea de total anual (m³/s)
fig.add_trace(
    go.Scatter(
        x=x, y=y,
        mode='markers',
        showlegend=False,
        marker=dict(color='black', size=10,symbol='triangle-up')
    ),
    secondary_y=True
)

# Línea LOWESS
fig.add_trace(
    go.Scatter(
        x=x, y=y_lowess,
        mode='lines',
        showlegend=False,
        line=dict(width=2,color='black')  # sin color fijo si no quieres
    ),
    secondary_y=True
)

# 3) Configuración de ejes
fig.update_yaxes(title_text="Percentage of Water Use [%]", secondary_y=False)
fig.update_yaxes(title_text="Water Use (m³/s)", secondary_y=True)
fig.update_xaxes(tickangle=45,tickfont=dict(size=16),
                 range=[1950, 2021]
                 )

# 4) Layout
fig.update_layout(
    barmode='stack',
    font=dict(size=18, color='black'),
    template="plotly_white",
    showlegend=False
)


fig.show()


In [ ]:
# NLWU
fig2 = make_subplots(specs=[[{"secondary_y": True}]])

# 1) Barras apiladas (100%) con porcentajes
for sector in sector_nlwu:
    fig2.add_trace(
        go.Bar(
            x=by_year_nlwu['Years'],
            y=by_year_nlwu[sector + '_pct'],
            name=sector,
            marker=dict(color=sector_colors.get(sector, None))  
        ),
        secondary_y=False
    )
fig2.update_traces(width=0.99,
                  marker=dict(line=dict(color='#000000',width=1))
                  )

# 2) Línea de total anual (m³/s)
fig2.add_trace(
    go.Scatter(
        x=x_nlwu, y=y_nlwu,
        mode='markers',
        showlegend=False,
        marker=dict(color='black', size=10,symbol='triangle-up')
    ),
    secondary_y=True
)

# Línea LOWESS
fig2.add_trace(
    go.Scatter(
        x=x_nlwu, y=y_lowess_nlwu,
        mode='lines',
        showlegend=False,
        line=dict(width=2,color='black')  # sin color fijo si no quieres
    ),
    secondary_y=True
)

# 3) Configuración de ejes
fig2.update_yaxes(title_text="Percentage of Water Use [%]", secondary_y=False)
fig2.update_yaxes(title_text="NLWU (m³/s)", secondary_y=True)
fig2.update_xaxes(tickangle=45,tickfont=dict(size=16),
                  range=[1949, 2021])

# 4) Layout
fig2.update_layout(
    barmode='stack',
    font=dict(size=18, color='black'),
    template="plotly_white",
    showlegend=False
)


fig2.show()


In [ ]:
! pip install --upgrade pip

In [ ]:
! pip install --upgrade kaleido

In [ ]:
import kaleido

In [ ]:
fig.write_image(imagen_path + "/WU_Chile.svg", width=1.5*600, height=0.75*600, scale=2)
fig2.write_image(imagen_path + "/NLWU_Chile.svg", width=1.5*600, height=0.75*600, scale=2)

## Agrupar / Group by

In [ ]:
#df_prueba = df.groupby('Years')['Value'].mean().reset_index()#reset index para que quede como dataframe
#df_prueba2 = df2.groupby('Years')['Value'].mean().reset_index()#reset index para que quede como dataframe
#df_prueba.head()

## NLWU Chile

In [ ]:
#Gráficos uso de agua con media móvil de 10 puntos
df_1_year = df.groupby('Years')['Value'].sum().reset_index()#reset index para que quede como dataframe
df_1_year.head()

fig1 = px.scatter(df_1_year, x="Years", y="Value",
                 labels={'Years': 'Año', 'Value': 'Uso hídrico [m³/s]'},
                 title="Uso de agua no asociado al suelo de Chile",
                 template='plotly_white',
                 trendline="rolling", trendline_options=dict(window=10))
fig1.update_traces(marker=dict(line=dict(color='#000000',width=1)))
fig1.update_layout(font=dict(size=16, color='black'))
fig1.show()


fig2 = px.scatter(df_1_year, x="Years", y="Value",
                 labels={'Years': 'Year', 'Value': 'Water use [m³/s]'},
                 title="Non Land Water Use in Chile",
                 template='plotly_white',
                 trendline="rolling", trendline_options=dict(window=10))
fig2.update_traces(marker=dict(line=dict(color='#000000',width=1)))
fig2.update_layout(font=dict(size=16, color='black'))
fig2.show()




In [ ]:
fig2 = px.scatter(df_1_year, x="Years", y="Value",
                 labels={'Years': 'Year', 'Value': 'Water use [m³/s]'},
                 title="Non Land Water Use in Chile",
                 template='plotly_white',
                 trendline="lowess",trendline_options=dict(frac=0.25))
fig2.update_traces(marker=dict(line=dict(color='#000000',width=1)))
fig2.update_layout(font=dict(size=16, color='black'))
fig2.show()

In [ ]:
# Guardar imágenes en svg

fig2.write_image(imagen_path + "/NLWU_Chile_en_lowess.svg", width=1.5*600, height=0.75*600, scale=2)

In [ ]:
#Gráficos de uso de agua con mínimos cuadrados ordinarios / Ordinary Least Squares regression for water use

df_1_year = df.groupby('Years')['Value'].sum().reset_index()#reset index para que quede como dataframe
df_1_year.head()

fig1 = px.scatter(df_1_year, x="Years", y="Value",
                 labels={'Years': 'Año', 'Value': 'Uso hídrico [m³/s]'},
                 title="Uso de agua no asociado al suelo de Chile",
                 template='plotly_white',
                 trendline="ols")
fig1.update_traces(marker=dict(line=dict(color='#000000',width=1)))
fig1.update_layout(font=dict(size=16, color='black'))
fig1.show()


fig2 = px.scatter(df_1_year, x="Years", y="Value",
                 labels={'Years': 'Year', 'Value': 'Water use [m³/s]'},
                 title="Non Land Water Use in Chile",
                 template='plotly_white',
                 trendline="ols")
fig2.update_traces(marker=dict(line=dict(color='#000000',width=1)))
fig2.update_layout(font=dict(size=16, color='black'))
fig2.show()


In [ ]:
# Guardar imagenes en png

fig1.write_image(imagen_path + "/NLWU_Chile_OLS_es.png", width=1.5*600, height=0.75*600, scale=2)

fig2.write_image(imagen_path + "/NLWU_Chile_OLS_en.png", width=1.5*600, height=0.75*600, scale=2)

# Guardar imágenes en svg

fig1.write_image(imagen_path + "/NLWU_Chile_OLS_es.svg")

fig2.write_image(imagen_path + "/NLWU_Chile_OLS_en.svg")

## NLWU Chile por sector

In [ ]:
# Agrupar el DataFrame por 'Level_1' y 'Years' y sumar los valores de 'Value'
grouped_df = df2.groupby(['Level_1', 'Years']) #esto el 14/05 no me resultó (no me dio igual que como lo tenía antes.. creo que hay algo que modifiqué o borré)
grouped_df.head()

In [ ]:
#En inglés
# Agrupar los usos de agua por sector y sumarlos
df2_grouped = df2.groupby('Level_1').agg({'Value': 'sum'}).reset_index()

# Crear un gráfico de torta con Plotly
fig = px.pie(df2_grouped, 
             values='Value', 
             names='Level_1', 
             #title='Distribución por sector del uso de agua no asociado al suelo de Chile',
             color='Level_1',
             color_discrete_map=colores_sector
                                 )

fig.update_traces(textinfo='text',  # Cambiar a 'text' para utilizar texttemplate
                  texttemplate='%{percent:.2%}',  # Formato de 2 decimales para los porcentajes
                  textposition='auto',
                  labels=df2_grouped['Level_1'].replace('DrinkingWater', 'Drinking Water'),
                  marker=dict(line=dict(color='#000000',width=1)))

fig.update_layout(font=dict(size=18, color='black'),
)
    

# Mostrar el gráfico
fig.show()




In [ ]:
# Guardar imagenes en png

fig.write_image(imagen_path + "/NLWU_Sector_en.png", width=1.5*600, height=1.5*600, scale=2)

# Guardar imágenes en svg

fig.write_image(imagen_path + "/NLWU_Sector_en.svg")



In [ ]:
#En español

fig2 =fig
# Reemplazar los labels de las trazas del gráfico
for trace in fig2.data:
    # Verifica si la traza tiene un atributo 'labels' (en gráficos de torta, esto debería ser 'names')
    if hasattr(trace, 'labels'):
        # Cambiar los nombres de los labels según el diccionario de reemplazo
        trace.labels = [english_to_spanish.get(label, label) for label in trace.labels]

# Mostrar el gráfico con los nombres de los labels en español
fig2.update_layout(font=dict(size=18, color='black'),
    #title='Distribución por sector del uso de agua no asociado al suelo de Chile'
    )
fig2.show()

# Guardar imagenes en png

fig2.write_image(imagen_path + "/NLWU_Sector_es.png", width=1.5*600, height=1.5*600, scale=2)

# Guardar imágenes en svg

fig2.write_image(imagen_path + "/NLWU_Sector_es.svg")

In [ ]:
# Gráfico para energía

#En inglés
df2_energy = df2.loc[df2['Level_1'] == 'Energy']
df2_energy_grouped = df2_energy.groupby('Level_2').agg({'Value': 'sum'}).reset_index()
df2_energy_grouped.head()

# Crear un gráfico de torta con Plotly
fig = px.pie(df2_energy_grouped, 
             values='Value', 
             names='Level_2', 
             #title='Distribución por sector del uso de agua no asociado al suelo de Chile',
             color='Level_2',
             color_discrete_map={'Hydraulic': '#377eb8',  # Asignar colores específicos por sector
                                 'NCRE': '#dede00',
                                 'Thermal': '#999999'},
                                 )

fig.update_traces(textinfo='text',  # Cambiar a 'text' para utilizar texttemplate
                  texttemplate='%{percent:.2%}',  # Formato de 2 decimales para los porcentajes
                  textposition='auto',
                  marker=dict(line=dict(color='#000000',width=1)))

fig.update_layout(font=dict(size=18, color='black'),
)
    

# Mostrar el gráfico
fig.show()



In [ ]:
# Guardar imagenes en png

fig.write_image(imagen_path + "/NLWU_energy_en.png", width=1.5*600, height=1.5*600, scale=2)

# Guardar imágenes en svg

fig.write_image(imagen_path + "/NLWU_energy_en.svg")


In [ ]:
#En español (copié la figura de arriba -en inglés-, ojo al correr)

# Definir un diccionario de reemplazo para cambiar de inglés a español
english_to_spanish = {
    'Hydraulic': 'Hidráulica',
    'NCRE': 'ERNC',
    'Thermal': 'Térmica',
}

fig2 =fig
# Reemplazar los labels de las trazas del gráfico
for trace in fig2.data:
    # Verifica si la traza tiene un atributo 'labels' (en gráficos de torta, esto debería ser 'names')
    if hasattr(trace, 'labels'):
        # Cambiar los nombres de los labels según el diccionario de reemplazo
        trace.labels = [english_to_spanish.get(label, label) for label in trace.labels]

# Mostrar el gráfico con los nombres de los labels en español
fig2.update_layout(font=dict(size=18, color='black'),

    )
fig2.show()


In [ ]:

# Guardar imagenes en png

fig2.write_image(imagen_path + "/NLWU_energy_es.png", width=1.5*600, height=1.5*600, scale=2)

# Guardar imágenes en svg

fig2.write_image(imagen_path + "/NLWU_energy_es.svg")

In [ ]:
#Con los gráficos que acabo de hacer, me da el paso a pasar directamente a NLWU excluyendo energía hidráulica

In [ ]:
# Crear una figura de subplots con 3 filas y 2 columnas (3x2)
fig = make_subplots(rows=3, cols=2, subplot_titles=[f'{level_1} Non Land Water Use in Chile' for level_1 in grouped_df['Level_1'].unique()] + ["Total Water Use in Chile"])

# Diccionario para mapear el nivel_1 a la posición de fila y columna en la figura
level_1_position = {
    'DrinkingWater': (1, 1),
    'Energy': (1, 2),
    'Industry': (2, 1),
    'Livestock': (2, 2),
    'Mining': (3, 1)
}

# Iterar sobre los grupos por 'Level_1' y añadir cada gráfico a la figura de subplots
for level_1, group in grouped_df.groupby('Level_1'):
    # Crear un scatter plot para cada sector (Level_1)
    scatter = px.scatter(group, x='Years', y='Value',
                         labels={'Years': 'Years', 'Value': 'Water use [m³/s]'},
                         trendline="rolling", trendline_options=dict(window=10))
    
    # Añadir el scatter plot como una traza a la figura de subplots
    # Obtenemos la fila y la columna correspondientes para el sector
    row, col = level_1_position[level_1]
    for trace in scatter.data:
        fig.add_trace(trace, row=row, col=col)

# Calcular el consumo total para cada año
consumo_total = df2.groupby('Years')['Value'].sum().reset_index()

# Crear un scatter plot para el consumo total
scatter_total = px.scatter(consumo_total, x='Years', y='Value',
                           labels={'Years': 'Years', 'Value': 'Water use [m³/s]'},
                           title='Total Non Land Water Use in Chile',trendline="rolling", trendline_options=dict(window=10))

# Añadir el scatter plot del consumo total como una traza a la figura de subplots en fila 3 columna 2
for trace in scatter_total.data:
    fig.add_trace(trace, row=3, col=2)

# Ajustar el diseño de la figura
fig.update_layout(height=1000, width=1000, title_text="Non Land Water Use in Chile for Different Sectors and Total")

# Mostrar la figura con subplots
fig.show()


In [ ]:
grouped_df.columns

In [ ]:
# Uso de agua en barras apiladas

# En inglés
grouped_df = df2.groupby(['Years', 'Level_1'])['Value'].sum().unstack(fill_value=0)
grouped_df = grouped_df.rename(columns=english_level)
# Crear un gráfico de barras acumuladas
fig = px.bar(grouped_df, x=grouped_df.index, y=grouped_df.columns, 
             labels={'value': 'Water Use [m³/s]', 'Years': 'Year'},
             #title='Non Land Water Use by Sector in Chile',
             template='plotly_white',
             barmode='stack',
             color='Level_1',
             color_discrete_map=colores_sector)

fig.update_traces(marker=dict(line=dict(color='#000000',width=1)))

fig.update_layout(
    font=dict(size=16, color='black'),  
    title=dict(font=dict(size=18)),  
    legend=dict(title_text=''))

fig.show()




In [ ]:
fig.write_image(imagen_path + "/NLWU_sectors_y_en.png", width=1.5*600, height=0.75*600, scale=2)

fig.write_image(imagen_path + "/NLWU_sectors_y_en.svg")

In [ ]:
# En Español
grouped_df_es = grouped_df.rename(columns=english_to_spanish)
fig = px.bar(grouped_df_es, x=grouped_df_es.index, y=grouped_df_es.columns, 
             labels={'value': 'Uso hídrico [m³/s]', 'Years': 'Año'},
             #title='Non Land Water Use by Sector in Chile',
             template='plotly_white',
             barmode='stack',
             color='Level_1',
             color_discrete_map=colores_sector)

fig.update_traces(marker=dict(line=dict(color='#000000',width=1)))

fig.update_layout(
    font=dict(size=16, color='black'),  
    title=dict(font=dict(size=18)),  
    legend=dict(title_text=''))

fig.show()


In [ ]:
fig.write_image(imagen_path + "/NLWU_sectors_y_es.png", width=1.5*600, height=0.75*600, scale=2)

fig.write_image(imagen_path + "/NLWU_sectors_y_es.svg")

### NLWU Chile por sector excluyendo energía hidráulica

In [ ]:
# Filtrar hydraulic
df2_filtered = df2.loc[df2['Level_2'] != 'Hydraulic']
grouped_df2 = df2_filtered.groupby(['Level_1','Level_2','Years']).agg({'Value': 'sum'}).reset_index()
grouped_df2.head()

In [ ]:
grouped_df2['Level_2'].unique()

In [ ]:
# Crear una figura de subplots con 3 filas y 2 columnas (3x2)
fig = make_subplots(rows=3, cols=2, subplot_titles=[f'{level_1} Non Land Water Use in Chile' for level_1 in grouped_df2['Level_1'].unique()] + ["Total Water Use in Chile"])

# Diccionario para mapear el nivel_1 a la posición de fila y columna en la figura
level_1_position = {
    'DrinkingWater': (1, 1),
    'Energy': (1, 2),
    'Industry': (2, 1),
    'Livestock': (2, 2),
    'Mining': (3, 1)
}

# Iterar sobre los grupos por 'Level_1' y añadir cada gráfico a la figura de subplots
for level_1, group in grouped_df2.groupby('Level_1'):
    # Calcular el consumo total por año para el grupo de Level_1
    consumo_annual = group.groupby('Years')['Value'].sum().reset_index()
    
    # Crear un scatter plot para cada sector (Level_1)
    scatter = px.scatter(consumo_annual, x='Years', y='Value',
                         labels={'Years': 'Years', 'Value': 'Water use [m³/s]'},
                         trendline="rolling", trendline_options=dict(window=10))
    
    # Añadir el scatter plot como una traza a la figura de subplots
    # Obtenemos la fila y la columna correspondientes para el sector
    row, col = level_1_position[level_1]
    for trace in scatter.data:
        fig.add_trace(trace, row=row, col=col)

# Calcular el consumo total para cada año en `df2_filtered`
consumo_total = df2_filtered.groupby('Years')['Value'].sum().reset_index()

# Crear un scatter plot para el consumo total
scatter_total = px.scatter(consumo_total, x='Years', y='Value',
                           labels={'Years': 'Years', 'Value': 'Water use [m³/s]'},
                           title='Total Non Land Water Use in Chile', trendline="rolling", trendline_options=dict(window=10))

# Añadir el scatter plot del consumo total como una traza a la figura de subplots en fila 3 columna 2
for trace in scatter_total.data:
    fig.add_trace(trace, row=3, col=2)

# Ajustar el diseño de la figura
fig.update_layout(height=1000, width=1000, title_text="Non Land Water Use in Chile for Different Sectors and Total. Consumptive use (excluding Hydraulic Energy)")

# Mostrar la figura con subplots
fig.show()


In [ ]:
grouped_df.head()

#### Gráfico de barras apiladas

In [ ]:
# En inglés

grouped_df = df2_filtered.groupby(['Years', 'Level_1'])['Value'].sum().unstack(fill_value=0) # Agrupar df2 por 'Years' y 'Level_1' y calcular la suma de 'Value'

fig = px.bar(grouped_df, x=grouped_df.index, y=grouped_df.columns, 
             labels={'value': 'Water Use [m³/s]', 'Years': 'Year'},
             color='Level_1',
             color_discrete_map={'DrinkingWater': '#377eb8',  # Asignar colores específicos por sector
                                 'Energy': '#dede00',
                                 'Industry': '#999999',
                                 'Livestock': '#f781bf',
                                 'Mining': '#ff7f00'},
             title='Non Land Water Use by Sector in Chile (excluding hydraulic energy)',
             template='plotly_white',
             barmode='stack')

fig.update_traces(marker=dict(line=dict(color='#000000',width=1)))

fig.update_layout(
    font=dict(size=16, color='black'),  
    title=dict(font=dict(size=18)),  
    legend=dict(title_text='')

)

for trace in fig.data: #arreglar Drinking Water para mostrarlo en la leyenda
    if trace.name == 'DrinkingWater':
        trace.name = 'Drinking Water'

fig.show()

# Guardar imagenes en png

fig.write_image(imagen_path + "/NLWU_sectors_exhydr_en.png", width=1.5*600, height=0.75*600, scale=2)

# Guardar imágenes en svg

fig.write_image(imagen_path + "/NLWU_sectors_exhydr_en.svg")


In [ ]:
# Para hacer el scatter filtrado (febrero 2025)
df_yearly = df2_filtered.groupby("Years", as_index=False)["Value"].sum()
df_yearly

fig2 = px.scatter(df_yearly, x="Years", y="Value",
                 labels={'Years': 'Year', 'Value': 'Water use [m³/s]'},
                 title="Non Land Water Use in Chile",
                 template='plotly_white',
                 trendline="lowess",trendline_options=dict(frac=0.1))
fig2.update_traces(marker=dict(line=dict(color='#000000',width=1)))
fig2.update_layout(font=dict(size=16, color='black'))
fig2.show()

In [ ]:
# Para hacer el scatter filtrado (febrero 2025)
df_yearly = df2_filtered.groupby("Years", as_index=False)["Value"].sum()
df_yearly

fig2 = px.scatter(df_yearly, x="Years", y="Value",
                 labels={'Years': 'Year', 'Value': 'Water use [m³/s]'},
                 title="Non Land Water Use in Chile",
                 template='plotly_white',
                 trendline="lowess",trendline_options=dict(frac=0.25))
fig2.update_traces(marker=dict(line=dict(color='#000000',width=1)))
fig2.update_layout(font=dict(size=16, color='black'))
fig2.show()

In [ ]:
fig2 = px.scatter(df_1_year, x="Years", y="Value",
                 labels={'Years': 'Year', 'Value': 'Water use [m³/s]'},
                 title="Non Land Water Use in Chile",
                 template='plotly_white',
                 trendline="rolling", trendline_options=dict(window=10))
fig2.update_traces(marker=dict(line=dict(color='#000000',width=1)))
fig2.update_layout(font=dict(size=16, color='black'))
fig2.show()

In [ ]:
# Guardar imágenes en svg

fig2.write_image(imagen_path + "/NLWU_scat_exhydr_en.svg", width=1.5*600, height=0.75*600, scale=2)

In [ ]:
import plotly.express as px
import pandas as pd

# Agrupar y calcular la suma por 'Years' y 'Level_1'
grouped_df = df2_filtered.groupby(['Years', 'Level_1'])['Value'].sum().unstack(fill_value=0)

# Normalizar para que cada año sume 100%
grouped_df_percent = grouped_df.div(grouped_df.sum(axis=1), axis=0) * 100

# Convertir a formato largo (melt)
df_long = grouped_df_percent.reset_index().melt(id_vars='Years', var_name='Sector', value_name='Percentage')

# Crear el gráfico de barras apiladas al 100%
fig = px.bar(
    df_long,
    x='Years',
    y='Percentage',
    color='Sector',  # Ahora 'Sector' es una única columna categórica
    labels={'Percentage': 'Percentage of Water Use [%]', 'Years': 'Year'},
    color_discrete_map={
        'DrinkingWater': '#377eb8',
        'Energy': '#dede00',
        'Industry': '#999999',
        'Livestock': '#f781bf',
        'Mining': '#ff7f00'
    },
    title='Non Land Water Use by Sector in Chile (excluding hydraulic energy)',
    template='plotly_white',
    barmode='stack'
)

# Mejorar el diseño
fig.update_traces(marker=dict(line=dict(color='#000000', width=1)))

fig.update_layout(
    font=dict(size=16, color='black'),
    title=dict(font=dict(size=18)),
    legend=dict(title_text=''),
    yaxis=dict(title='Percentage of Water Use [%]', range=[0, 100])
)

# Ajustar nombre en la leyenda
for trace in fig.data:
    if trace.name == 'DrinkingWater':
        trace.name = 'Drinking Water'

fig.show()



In [ ]:
# Guardar imágenes en svg
fig.write_image(imagen_path + "/NLWU_sect_exhydr_agrup.svg", width=1.5*600, height=0.75*600, scale=2)

In [ ]:
# En español

grouped_df = df2_filtered.groupby(['Years', 'Level_1'])['Value'].sum().unstack(fill_value=0) # Agrupar df2 por 'Years' y 'Level_1' y calcular la suma de 'Value'

fig2 = px.bar(grouped_df, x=grouped_df.index, y=grouped_df.columns, 
             title='Uso de agua de Chile no asociado al suelo, por sector (excluyendo energía hidráulica)',
             labels={
                 'value': 'Uso de agua [m³/s]',
                 'Years': 'Año'
             },
            color='Level_1',
             color_discrete_map={'DrinkingWater': '#377eb8',  # Asignar colores específicos por sector
                                 'Energy': '#dede00',
                                 'Industry': '#999999',
                                 'Livestock': '#f781bf',
                                 'Mining': '#ff7f00'},
            template='plotly_white',
             barmode='stack')

# Cambiar el color de los sectores
fig2.update_traces(marker=dict(line=dict(color='#000000', width=1)))

# Ajustar configuraciones del diseño del gráfico
fig2.update_layout(
    font=dict(size=16, color='black'),
    title=dict(font=dict(size=18)),
    yaxis_title='Uso de agua [m³/s]',
    xaxis_title='Año',
    legend=dict(
        title_text='',
        font=dict(size=16)
    )
)

# Diccionario para mapear los nombres originales a los nombres en español
sector_name_mapping = {
    'DrinkingWater': 'Agua Potable',
    'Energy': 'Energía',
    'Industry': 'Industria',
    'Livestock': 'Pecuario',
    'Mining': 'Minería'
}

# Recorrer las trazas y cambiar sus nombres según el mapeo de nombres
for trace in fig2.data:
    if trace.name in sector_name_mapping:
        trace.name = sector_name_mapping[trace.name]

# Mostrar el gráfico con la leyenda actualizada
fig2.show()

# Guardar imagenes en png

fig2.write_image(imagen_path + "/NLWU_sectors_exhydr_es.png", width=1.5*600, height=0.75*600, scale=2)

# Guardar imágenes en svg

fig2.write_image(imagen_path + "/NLWU_sectors_exhydr_es.svg")


In [ ]:
grouped_df

In [ ]:
grouped_df.loc[1990]

## NLWU Chile por macrozona

In [ ]:
df.head()

In [ ]:
#Agrupar por región
df_1_reg = df.groupby(['nom_reg','Years'])['Value'].sum().reset_index()#reset index para que quede como dataframe
df_1_reg.head()



In [ ]:
df2_filtered.head()

In [ ]:
#Arupado sin energía hidráulica
df2_reg_filt = df2_filtered.groupby(['nom_reg','Years'])['Value'].sum().reset_index()#reset index para que quede como dataframe
df2_reg_filt.head()

In [ ]:
#Asignar macrozonas
# Crear un diccionario para mapear cada región a su respectiva macrozona
regiones_a_macrozona = {
    'REGIÓN DE ANTOFAGASTA': 'Macrozona Norte',
    'REGIÓN DE ARICA Y PARINACOTA': 'Macrozona Norte',
    'REGIÓN DE ATACAMA': 'Macrozona Norte',
    'REGIÓN DE AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO': 'Macrozona Austral',
    'REGIÓN DE COQUIMBO': 'Macrozona Centro',
    'REGIÓN DE LA ARAUCANÍA': 'Macrozona Sur',
    'REGIÓN DE LOS LAGOS': 'Macrozona Sur',
    'REGIÓN DE LOS RÍOS': 'Macrozona Sur',
    'REGIÓN DE MAGALLANES Y DE LA ANTÁRTICA CHILENA': 'Macrozona Austral',
    'REGIÓN DE TARAPACÁ': 'Macrozona Norte',
    'REGIÓN DE VALPARAÍSO': 'Macrozona Centro',
    'REGIÓN DE ÑUBLE': 'Macrozona Centro Sur',
    'REGIÓN DEL BIOBÍO': 'Macrozona Centro Sur',
    "REGIÓN DEL LIBERTADOR GENERAL BERNARDO O'HIGGINS": 'Macrozona Centro Sur',
    'REGIÓN DEL MAULE': 'Macrozona Centro Sur',
    'REGIÓN METROPOLITANA DE SANTIAGO': 'Macrozona Centro'
}

# Asignar una nueva columna 'Macrozona' basada en el diccionario
df_1_reg['Macrozona'] = df_1_reg['nom_reg'].map(regiones_a_macrozona)
df2_reg_filt['Macrozona'] = df2_reg_filt['nom_reg'].map(regiones_a_macrozona)


In [ ]:
# Ordenar las macrozonas en el orden deseado con nombres completos
#orden_macrozonas = ['Macrozona Norte', 'Macrozona Centro', 'Macrozona Centro Sur', 'Macrozona Sur', 'Macrozona Austral']
orden_macrozonas = ['Northern Macrozone', 'Central Macrozone', 'Central-Southern Macrozone', 'Southern Macrozone', 'Austral Macrozone']
# Agrupar por 'Years' y 'macrozona' y calcular la suma de 'Value' para cada grupo
grouped_df = df_1_reg.groupby(['Years', 'Macrozona'])['Value'].sum().reset_index()

# Transformar el DataFrame agrupado para que las macrozonas sean columnas usando unstack
grouped_df_unstacked = grouped_df.set_index(['Years', 'Macrozona']).unstack()
grouped_df_unstacked = grouped_df_unstacked.rename(columns=spanish_to_english)

# Restablecer el índice
grouped_df_unstacked.columns = grouped_df_unstacked.columns.droplevel()  # Aplanar multiíndice en columnas

# Reordenar las columnas del DataFrame según el orden deseado de las macrozonas
grouped_df_unstacked = grouped_df_unstacked[orden_macrozonas]

# Crear un gráfico de barras acumulado usando plotly.express
fig = px.bar(
    grouped_df_unstacked,
    x=grouped_df_unstacked.index,  # Eje x con los años
    y=orden_macrozonas,  # Eje y con las macrozonas como columnas ordenadas
    labels={'value': 'Water Use [m³/s]', 'x': 'Year'},  # Etiquetas de los ejes
    title='Non Land Water Use by Macrozone in Chile',
    color_discrete_map={
        'Northern Macrozone': '#ff9896',  # Asignar colores específicos a cada macrozona
        'Central Macrozone': '#dbdb8d',
        'Central-Southern Macrozone': '#98df8a',
        'Southern Macrozone': '#9edae5',
        'Austral Macrozone': '#c5b0d5'
    },
    template='plotly_white',
)
fig.update_layout(
    font=dict(size=16, color='black'),  # Tamaño y color de la fuente
    title=dict(font=dict(size=18)),  # Tamaño de la fuente del título a 18
    #yaxis_title='Uso de agua [m³/s]',  # Cambiar el título del eje y
    #xaxis_title='Año',  # Cambiar el título del eje x
    legend=dict(
        title_text='',  # Eliminar el título de la leyenda
        font=dict(size=16)  # Tamaño de la fuente de la leyenda
    )
)
# Actualizar el estilo de las trazas
fig.update_traces(marker=dict(line=dict(color='#000000', width=0.5)))

# Mostrar el gráfico
fig.show()


In [ ]:
#ESTE GRÁFICO NO ME APORTA MUCHO, ME QUEDARÉ CON EL DE BARRAS APILADAS (BORRAR?)
# Agrupar el dataframe df_1_reg por 'Macrozona'
grouped_macrozona = df_1_reg.groupby('Macrozona')

# Crear una figura de subplots con 3 filas y 2 columnas (3x2)
fig = make_subplots(rows=3, cols=2, subplot_titles=[f'{macrozona} Non Land Water Use in Chile' for macrozona in grouped_macrozona.groups.keys()] + ["Total Water Use in Chile"])

# Diccionario para mapear las macrozonas a la posición de fila y columna en la figura
macrozona_position = {
    'Macrozona Norte': (1, 1),
    'Macrozona Centro': (1, 2),
    'Macrozona Centro Sur': (2, 1),
    'Macrozona Sur': (2, 2),
    'Macrozona Austral': (3, 1)
}

# Iterar sobre los grupos por 'Macrozona' y añadir cada gráfico a la figura de subplots
for macrozona, group in grouped_macrozona:
    # Calcular el consumo total por año para el grupo de Macrozona
    consumo_annual = group.groupby('Years')['Value'].sum().reset_index()
    
    # Crear un scatter plot para cada macrozona
    scatter = px.scatter(consumo_annual, x='Years', y='Value',
                         labels={'Years': 'Years', 'Value': 'Water use [m³/s]'},
                         trendline="rolling", trendline_options=dict(window=10))
    
    # Añadir el scatter plot como una traza a la figura de subplots
    # Obtenemos la fila y la columna correspondientes para la macrozona
    row, col = macrozona_position[macrozona]
    for trace in scatter.data:
        fig.add_trace(trace, row=row, col=col)

# Calcular el consumo total para cada año en df2_filtered
consumo_total = df2_filtered.groupby('Years')['Value'].sum().reset_index()

# Crear un scatter plot para el consumo total
scatter_total = px.scatter(consumo_total, x='Years', y='Value',
                           labels={'Years': 'Years', 'Value': 'Water use [m³/s]'},
                           title='Total Non Land Water Use in Chile', trendline="rolling", trendline_options=dict(window=10))

# Añadir el scatter plot del consumo total como una traza a la figura de subplots en fila 3 columna 2
for trace in scatter_total.data:
    fig.add_trace(trace, row=3, col=2)

# Ajustar el diseño de la figura
fig.update_layout(height=1000, width=1000, title_text="Non Land Water Use in Chile for Different Macrozones and Total")

# Mostrar la figura con subplots
fig.show()


### NLWU Chile por macrozona excluyendo energía hidráulica

In [ ]:
df.head()

In [ ]:
# Español

# Ordenar las macrozonas en el orden deseado con nombres completos
orden_macrozonas = ['Macrozona Norte', 'Macrozona Centro', 'Macrozona Centro Sur', 'Macrozona Sur', 'Macrozona Austral']
#orden_macrozonas = ['Northern Macrozone', 'Central Macrozone', 'Central-Southern Macrozone', 'Southern Macrozone', 'Austral Macrozone']
# Agrupar por 'Years' y 'macrozona' y calcular la suma de 'Value' para cada grupo
grouped_df = df2_reg_filt.groupby(['Years', 'Macrozona'])['Value'].sum().reset_index()

# Transformar el DataFrame agrupado para que las macrozonas sean columnas usando unstack
grouped_df_unstacked = grouped_df.set_index(['Years', 'Macrozona']).unstack()
#grouped_df_unstacked = grouped_df_unstacked.rename(columns=spanish_to_english)

# Restablecer el índice
grouped_df_unstacked.columns = grouped_df_unstacked.columns.droplevel()  # Aplanar multiíndice en columnas

# Reordenar las columnas del DataFrame según el orden deseado de las macrozonas
grouped_df_unstacked = grouped_df_unstacked[orden_macrozonas]

# Crear un gráfico de barras acumulado usando plotly.express
fig = px.bar(
    grouped_df_unstacked,
    x=grouped_df_unstacked.index,  # Eje x con los años
    y=orden_macrozonas,  # Eje y con las macrozonas como columnas ordenadas
    labels={'value': 'Uso de agua [m³/s]', 'x': 'Año'},  # Etiquetas de los ejes
    title='Uso de Agua no asociado al suelo, por macrozona (excluyendo energía hidráulica)',
    color_discrete_map={
        'Macrozona Norte': '#ff9896',  # Asignar colores específicos a cada macrozona
        'Macrozona Centro': '#dbdb8d',
        'Macrozona Centro Sur': '#98df8a',
        'Macrozona Sur': '#9edae5',
        'Macrozona Austral': '#c5b0d5'
    },
    template='plotly_white',
)
fig.update_layout(
    font=dict(size=16, color='black'),  # Tamaño y color de la fuente
    title=dict(font=dict(size=18)),  # Tamaño de la fuente del título a 18
    #yaxis_title='Uso de agua [m³/s]',  # Cambiar el título del eje y
    #xaxis_title='Año',  # Cambiar el título del eje x
    legend=dict(
        title_text='',  # Eliminar el título de la leyenda
        font=dict(size=16)  # Tamaño de la fuente de la leyenda
    )
)
# Actualizar el estilo de las trazas
fig.update_traces(marker=dict(line=dict(color='#000000', width=0.5)))

# Mostrar el gráfico
fig.show()

# Guardar imagenes en png

fig.write_image(imagen_path + "/NLWU_macroz_exhydr_es.png", width=1.5*600, height=0.75*600, scale=2)

# Guardar imágenes en svg
fig.write_image(imagen_path + "/NLWU_macroz_exhydr_es.svg")

In [ ]:
import plotly.express as px
import pandas as pd

# Definir el orden de las macrozonas
orden_macrozonas = ['Macrozona Norte', 'Macrozona Centro', 'Macrozona Centro Sur', 'Macrozona Sur', 'Macrozona Austral']

# Agrupar por 'Years' y 'Macrozona' y calcular la suma de 'Value'
grouped_df = df2_reg_filt.groupby(['Years', 'Macrozona'])['Value'].sum().unstack(fill_value=0)

# Normalizar para que cada año sume 100%
grouped_df_percent = grouped_df.div(grouped_df.sum(axis=1), axis=0) * 100

# Reordenar columnas en el orden deseado
grouped_df_percent = grouped_df_percent[orden_macrozonas]

# Convertir a formato largo
df_long = grouped_df_percent.reset_index().melt(id_vars='Years', var_name='Macrozona', value_name='Percentage')

# Crear el gráfico de barras apiladas al 100%
fig = px.bar(
    df_long,
    x='Years',
    y='Percentage',
    color='Macrozona',
    labels={'Percentage': 'Porcentaje de Uso de Agua [%]', 'Years': 'Año'},
    title='Uso de Agua no asociado al suelo, por macrozona (excluyendo energía hidráulica)',
    color_discrete_map={
        'Macrozona Norte': '#ff9896',
        'Macrozona Centro': '#dbdb8d',
        'Macrozona Centro Sur': '#98df8a',
        'Macrozona Sur': '#9edae5',
        'Macrozona Austral': '#c5b0d5'
    },
    template='plotly_white',
    barmode='stack'
)

# Mejorar el diseño
fig.update_traces(marker=dict(line=dict(color='#000000', width=0.5)))

fig.update_layout(
    font=dict(size=16, color='black'),
    title=dict(font=dict(size=18)),
    legend=dict(title_text='', font=dict(size=16)),
    yaxis=dict(title='Porcentaje de Uso de Agua [%]', range=[0, 100])
)

# Mostrar el gráfico
fig.show()

# # Guardar imagenes en png
# fig.write_image(imagen_path + "/NLWU_macroz_exhydr_es.png", width=1.5*600, height=0.75*600, scale=2)

# # Guardar imágenes en svg
# fig.write_image(imagen_path + "/NLWU_macroz_exhydr_es.svg")


In [ ]:
# Guardar imágenes en svg
fig.write_image(imagen_path + "/NLWU_macroz_exhydr_agrup.svg", width=1.5*600, height=0.75*600, scale=2)


### NLWU por macrozona en mapa

In [ ]:
df2.head()

In [ ]:
#Los dataframe que voy a plotear son
#df_1_reg.head() #Agrupados por región
#df2_reg_filt.head() #Agrupados por región pero sin hidráulica

#los tuve que agrupar de nuevo, dejando el cod_reg... estos son los promedios de uso de agua anual
df_reg_map = df.groupby(['nom_reg','cod_reg'])['Value'].mean().reset_index()
df_reg__filt_map = df2_filtered.groupby(['nom_reg','cod_reg'])['Value'].mean().reset_index()

In [ ]:
df2_filtered

In [ ]:
pivot_table_f

In [ ]:
df_reg__filt_map.head()

In [ ]:
#Leer geodatos
reg = gpd.read_file(path+'/CR2/GeoJSON/regions_ContinentalChile_epsg4326.geojson') #la columna id de reg tiene los identificadores de las regiones, los mismos que cod_reg


In [ ]:
reg['id'] = reg['id'].astype(int) #dejar la columna de id como enteros

In [ ]:
reg.sample(n=10)

In [ ]:
reg.id

In [ ]:
#Para poder plotear con choropleth, establecer columna id como índice
reg.set_index('id', inplace=True)

In [ ]:
# Crear un mapa coroplético básico con el GeoDataFrame 'reg' PARA VERIFICAR QUE ESTÁN BUEN LAS REGIONES
fig = px.choropleth(
    reg,
    geojson=reg.geometry,
    locations=reg.index,  # Usar el índice de 'reg' como ubicaciones
    color=reg.index,  # Usar el índice de 'reg' para asignar colores
    title='Mapa de las Regiones de Chile',
)

# Ajustar el mapa para encajar los límites geográficos
fig.update_geos(fitbounds="geojson", visible=False)

# Mostrar el mapa
fig.show()


In [ ]:
#Unir geodataframe a dataframe
merged_gdf = reg.merge(df_reg_map, left_on='id', right_on='cod_reg')
merged_gdf_filt = reg.merge(df_reg__filt_map, left_on='id', right_on='cod_reg')

#Para poder plotear con choropleth, establecer columna id como índice
merged_gdf.set_index('cod_reg', inplace=True)
merged_gdf_filt.set_index('cod_reg', inplace=True)

In [ ]:
merged_gdf

In [ ]:
prueba = merged_gdf[merged_gdf['nom_reg'] == 'REGIÓN DE TARAPACÁ']

In [ ]:
pip install plotly --upgrade


In [ ]:
merged_gdf.head()

In [ ]:

#merged_gdf.index = merged_gdf.index + 1 #al hacer que partiera desde 1 me graficó la región de tarapacá pero me dejó de graficar otras :/
# Crear un mapa coroplético con el GeoDataFrame 
fig = px.choropleth(
    merged_gdf_filt,
    geojson=merged_gdf_filt.geometry,
    locations=merged_gdf_filt.id,  # Usar el código de región como ubicaciones
    color=merged_gdf_filt.Value,  # Usar el valor para asignar colores
    hover_data=['nom_reg', 'Value'],  # Mostrar información adicional al pasar el cursor
    #title='Mapa de las Regiones de Chile',
)

# Ajustar el mapa para encajar los límites geográficos
fig.update_geos(projection_scale=5, center_lon=-70, center_lat=-20, fitbounds="geojson", visible=False)


# Mostrar el mapa
fig.show()


In [ ]:
# ESTE ES UN INTENTO DE PONER LÍNEAS DE LAT Y LONG

# Asegurarte que el índice de merged_gdf esté correcto
merged_gdf.index = merged_gdf.index + 1

# Crear un mapa coroplético con el GeoDataFrame
fig = px.choropleth(
    merged_gdf,
    geojson=merged_gdf.geometry,
    locations='cod_reg',  # Usar el código de región como ubicaciones
    color='Value',  # Usar el valor para asignar colores
    hover_data=['nom_reg', 'cod_reg', 'Value'],  # Mostrar información adicional al pasar el cursor
    #title='Mapa de las Regiones de Chile',
)

# Añadir líneas de grilla de latitud y longitud
latitudes = [-18, -22, -26, -30, -34, -38, -42, -46, -50, -54, -60]
longitudes = [-74, -72, -70, -68, -66]

# Agregar líneas de latitud
for lat in latitudes:
    fig.add_trace(
        go.Scattergeo(
            lon=[merged_gdf.total_bounds[0], merged_gdf.total_bounds[2]],
            lat=[lat, lat],
            mode='lines',
            line=dict(width=1, color='black'),
            showlegend=False
        )
    )

# Agregar líneas de longitud
for lon in longitudes:
    fig.add_trace(
        go.Scattergeo(
            lon=[lon, lon],
            lat=[merged_gdf.total_bounds[1], merged_gdf.total_bounds[3]],
            mode='lines',
            line=dict(width=1, color='black'),
            showlegend=False
        )
    )

# Ajustar el mapa para encajar los límites geográficos
fig.update_geos(projection_scale=5, center_lon=-70, center_lat=-20, fitbounds="geojson", visible=False)

# Mostrar el mapa
fig.show()


In [ ]:
print(merged_gdf.crs) #verificar el sistema de coordinadas. WGS 84 (EPSG 4326)


#### Gráficos en Matplotlib


In [ ]:
#Gráfico con matplotlib de consumo hídrico total por región

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

merged_gdf_filt.plot(column='Value', cmap='viridis', linewidth=0.6, 
                    #edgecolor='black',
                    ax=ax, legend=True)

# Añadir un título al gráfico
#ax.set_title('Consumo hídrico por región', fontsize=16)

#plt.colorbar(label=r'NLWU [m³/s]')

# Mostrar el mapa
plt.show()



In [ ]:
# Suponiendo que merged_gdf_filt ya esté definido y contenga los datos
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Crear el mapa
cmap = plt.get_cmap('viridis')
norm = plt.Normalize(vmin=merged_gdf_filt['Value'].min(), vmax=merged_gdf_filt['Value'].max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

merged_gdf_filt.plot(column='Value', cmap='viridis', linewidth=0.6, ax=ax, legend=False)

# Añadir la barra de color con un título
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('Consumo hídrico [NLWU m³/s]', fontsize=12, labelpad=20)
cbar.ax.xaxis.set_label_position('top')
cbar.ax.xaxis.set_ticks_position('top')

# Añadir un título al gráfico
ax.set_title('Uso de agua medio anual (1950-2020)', fontsize=10)

# Mostrar el mapa
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

# Supongamos que merged_gdf_filt ya esté definido y contenga los datos

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Crear el mapa
cmap = plt.get_cmap('viridis')
norm = plt.Normalize(vmin=merged_gdf_filt['Value'].min(), vmax=merged_gdf_filt['Value'].max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

merged_gdf_filt.plot(column='Value', cmap='viridis', linewidth=0.6, ax=ax, legend=False)

# Añadir la barra de color
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('Consumo hídrico [NLWU m³/s]', fontsize=12)

# Mover el título de la barra de color arriba de la barra
cbar.ax.yaxis.set_label_position('right')
cbar.ax.yaxis.set_ticks_position('right')

cbar.ax.yaxis.labelpad = 20  # Ajusta la distancia del título

# Añadir un título al gráfico
ax.set_title('Consumo hídrico por región', fontsize=16)

# Mostrar el mapa
plt.show()



In [ ]:
# Suponiendo que merged_gdf_filt ya esté definido y contenga los datos
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Crear el mapa
cmap = plt.get_cmap('viridis')
norm = plt.Normalize(vmin=merged_gdf_filt['Value'].min(), vmax=merged_gdf_filt['Value'].max())
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

merged_gdf_filt.plot(column='Value', cmap='viridis', linewidth=0.6, ax=ax, legend=False)

# Añadir la barra de color con un título
cbar = fig.colorbar(sm, ax=ax)
#cbar.set_label('Consumo hídrico [m³/s]', fontsize=12)

# Añadir un título al gráfico
ax.set_title('Annual average water use (1950-2020)', fontsize=10)

# Mostrar el mapa
plt.show()


In [ ]:
#Gráfico con matplotlib de consumo hídrico total por región

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

merged_gdf.plot(column='Value', cmap='YlGnBu', linewidth=0.8, edgecolor='0.8', ax=ax, legend=True)

# Añadir un título al gráfico
ax.set_title('Non Land Water Use in Chile (excluding hydraulic energy)', fontsize=16)

# Mostrar el mapa
plt.show()


#### Gráficos en Plotly

In [ ]:
#print(merged_gdf[['id', 'Value', 'geometry']].head())


In [ ]:
# Crear un mapa coroplético básico con el GeoDataFrame 'merged_gdf'
fig = px.choropleth(
    merged_gdf,
    geojson=merged_gdf.geometry,
    locations=merged_gdf.index,  # Usar 'id' como locations, ya que el índice de merged_gdf es 'id'
    color='Value',  # Usar la columna 'Value' para asignar colores
    color_continuous_scale='YlGnBu',  # Escala de color continua
    #title='Non Land Water Use in Chile (excluding hydraulic energy)'
)

# Ajustar el mapa para encajar los límites geográficos
fig.update_geos(fitbounds="geojson", visible=True, 
                showframe=True,
                lataxis_showgrid=True, lonaxis_showgrid=True,
                lataxis_range=[-56, -17], # Límites de latitud para Chile continental
                lonaxis_range=[-76, -66], # Límites de longitud para Chile continental
                subunitwidth=0.5  
                #lataxis=dict(tick0=-54,dtick=5), #ESTOY TRATANDO DE QUE APAREZCAN LAS LAT Y LONG, PERO NO LO LOGRO)
                #lonaxis=dict(tick0=-74,dtick=5)

                )


fig.update_layout(
    coloraxis_colorbar=dict(title="Uso de agua [m³/s]"),
        margin=dict(
        l=20,  # margen izquierdo
        r=20,  # margen derecho
        t=30,  # margen superior
        b=10,  # margen inferior
    ),
    legend=dict(
        yanchor="top",
        y=0.5,  # posición vertical de la leyenda (cerca del borde superior)
        xanchor="right",
        x=0.5,  # posición horizontal de la leyenda (cerca del borde derecho)
        font=dict(size=10),  # tamaño de la fuente de la leyenda
    )

    #ESTOY TRATANDO DE QUE APAREZCAN LAS LAT Y LONG, PERO NO LO LOGRO)
    # geo=dict(
    #     lataxis=dict(
    #         showgrid=True,  # Mostrar la cuadrícula
    #         tickmode='array',
    #         tickvals=[-54, -50, -45, -40, -35, -30, -25, -20, -14],  # Valores para las marcas de latitud
    #         ticktext=['-54°', '-50°', '-45°', '-40°', '-35°', '-30°', '-25°', '-20°', '-14°'],  # Textos de los ticks
    #     ),
    #     lonaxis=dict(
    #         showgrid=True,  # Mostrar la cuadrícula
    #         tickmode='array',
    #         tickvals=[-75, -74, -73, -72, -71, -70, -69, -68, -67, -66],  # Valores para las marcas de longitud
    #         ticktext=['-75°', '-74°', '-73°', '-72°', '-71°', '-70°', '-69°', '-68°', '-67°', '-66°'],  # Textos de los ticks
    #     )
    # )
)

# Mostrar el mapa
fig.show()


In [ ]:
# Guardar imagenes en png TODAVÍA NO PUEDO HACER QUE QUEDEN LINDAS

fig.write_image(imagen_path + "/NLWU_reg_exhydr_map_es.png", width=0.5*600, height=1*600, scale=2)

# Guardar imágenes en svg
fig.write_image(imagen_path + "/NLWU_reg_exhydr_map_es.svg")

# NLWU 6 Ciudades

In [ ]:
pivot_table_f.head()

In [ ]:
pivot_table_f_2020 = pivot_table_f.loc[pivot_table_f['Years'] == 2020]
pivot_table_f_2020.head()

In [ ]:
# Ciudades

#  Iquique
f_2020_iquique = pivot_table_f_2020.loc[pivot_table_f_2020['nom_com'].isin(['IQUIQUE', 'ALTO HOSPICIO'])]

# Valparaíso
f_2020_valpo = pivot_table_f_2020.loc[pivot_table_f_2020['nom_com'].isin(['VALPARAÍSO', 'VIÑA DEL MAR', 'CONCÓN','QUILPUÉ','VILLA ALEMANA'])]

# Concepción
f_2020_conce = pivot_table_f_2020.loc[pivot_table_f_2020['nom_com'].isin(['TOMÉ', 'PENCO', 'TALCAHUANO','CONCEPCIÓN','HUALPÉN','CHIGUAYANTE','HUALQUI','SANTA JUANA','SAN PEDRO DE LA PAZ','CORONEL','LOTA'])]

# Valdivia
f_2020_valdivia = pivot_table_f_2020.loc[pivot_table_f_2020['nom_com']=='VALDIVIA']

# Puerto Montt
f_2020_puerto = pivot_table_f_2020.loc[pivot_table_f_2020['nom_com'].isin(['PUERTO MONTT','PUERTO VARAS'])]

In [ ]:
# Ciudades

# Santiago
comunas = ['Cerrillos','Cerro Navia','Conchalí','El Bosque','Estación Central','Huechuraba','Independencia','La Cisterna','La Florida','La Granja','La Pintana',
'La Reina','Las Condes','Lo Barnechea','Lo Espejo','Lo Prado','Macul','Maipú','Ñuñoa','Pedro Aguirre Cerda','Peñalolén','Providencia','Pudahuel',
'Puente Alto','Quilicura','Quinta Normal','Recoleta','Renca','San Bernardo','San Joaquín','San Miguel','San Ramón','Santiago','Vitacura']

for i in range(len(comunas)):
    comunas[i] = comunas[i].upper()
print(comunas)

f_2020_stgo = pivot_table_f_2020.loc[pivot_table_f_2020['nom_com'].isin(comunas)]
f_2020_stgo

In [ ]:
f_2020_stgo.shape

In [ ]:
colores_sector

In [ ]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots

# Colores de los sectores
colores_sector = {
    'DrinkingWater': '#377eb8',
    'Energy': '#dede00',
    'Industry': '#999999',
    'Livestock': '#f781bf',
    'Mining': '#ff7f00' 
}

# Supongamos que f_2020_iquique es un DataFrame que ya tienes
f_2020_iquique_total = f_2020_iquique[['DrinkingWater', 'Energy', 'Industry', 'Livestock', 'Mining']].sum()

data1 = pd.DataFrame({
    'Sector': f_2020_iquique_total.index,
    'Consumption': f_2020_iquique_total.values
})

fig1 = px.pie(data1, names='Sector', values='Consumption', color='Sector',color_discrete_map=colores_sector)
fig1.update_traces(textfont_size=12, marker=dict(line=dict(color='#000000', width=2)))
fig1.update_layout(uniformtext_minsize=12, uniformtext_mode='hide', font=dict(size=12, color='black'))
fig1.show()


In [ ]:
# Gráficos

colores_sector = {
    'DrinkingWater': '#377eb8',
    'Energy': '#dede00',
    'Industry': '#999999',
    'Livestock': '#f781bf',
    'Mining': '#ff7f00' 
}

# Iquique
f_2020_iquique_total = f_2020_iquique[['DrinkingWater', 'Energy', 'Industry', 'Livestock', 'Mining']].sum()

data1 = pd.DataFrame({
    'Sector': f_2020_iquique_total.index,
    'Consumption': f_2020_iquique_total.values
})

fig1 = px.pie(data1, names='Sector', values='Consumption',color = 'Sector', color_discrete_map=colores_sector)

# Valparaíso
f_2020_valpo_total = f_2020_valpo[['DrinkingWater', 'Energy', 'Industry', 'Livestock', 'Mining']].sum()

data2 = pd.DataFrame({
    'Sector': f_2020_valpo_total.index,
    'Consumption': f_2020_iquique_total.index
})

fig2 = px.pie(data2, names='Sector', values='Consumption',color = 'Sector',color_discrete_map=colores_sector)

# Santiago
f_2020_stgo_total = f_2020_stgo[['DrinkingWater', 'Energy', 'Industry', 'Livestock', 'Mining']].sum()

data3 = pd.DataFrame({
    'Sector': f_2020_stgo_total.index,
    'Consumption': f_2020_stgo_total.values
})

fig3 = px.pie(data3, names='Sector', values='Consumption',color = 'Sector',color_discrete_map=colores_sector)

# Concepción
f_2020_conce_total = f_2020_conce[['DrinkingWater', 'Energy', 'Industry', 'Livestock', 'Mining']].sum()

data4 = pd.DataFrame({
    'Sector': f_2020_conce_total.index,
    'Consumption': f_2020_conce_total.values
})

fig4 = px.pie(data4, names='Sector', values='Consumption',color = 'Sector',color_discrete_map=colores_sector)

# Valdivia
f_2020_valdi_total = f_2020_valdivia[['DrinkingWater', 'Energy', 'Industry', 'Livestock', 'Mining']].sum()

data5 = pd.DataFrame({
    'Sector': f_2020_valdi_total.index,
    'Consumption': f_2020_valdi_total.values
})

fig5 = px.pie(data5, names='Sector', values='Consumption',color = 'Sector',color_discrete_map=colores_sector)

# Puerto Montt
f_2020_puerto_total = f_2020_puerto[['DrinkingWater', 'Energy', 'Industry', 'Livestock', 'Mining']].sum()

data6 = pd.DataFrame({
    'Sector': f_2020_puerto_total.index,
    'Consumption': f_2020_puerto_total.values
})

fig6 = px.pie(data6, names='Sector', values='Consumption',color = 'Sector',color_discrete_map=colores_sector)


# Crear subplots especificando que son gráficos de pastel
fig = make_subplots(rows=1, cols=6, 
                    subplot_titles=("Iquique", "Valparaíso", "Santiago","Concepción","Valdivia","Puerto Montt"),
                    specs=[[{'type': 'pie'}, {'type': 'pie'}, {'type': 'pie'},
                            {'type': 'pie'}, {'type': 'pie'}, {'type': 'pie'}]])

# Añadir los gráficos a las subplots
fig.add_trace(fig1['data'][0], row=1, col=1)
fig.add_trace(fig2['data'][0], row=1, col=2)
fig.add_trace(fig3['data'][0], row=1, col=3)
fig.add_trace(fig4['data'][0], row=1, col=4)
fig.add_trace(fig5['data'][0], row=1, col=5)
fig.add_trace(fig6['data'][0], row=1, col=6)

# Actualizar el diseño
fig.update_traces(textfont_size=12, textinfo='percent',texttemplate='%{percent:.1%}', marker=dict(line=dict(color='#000000', width=1.5)))
fig.update_layout(uniformtext_minsize=12, uniformtext_mode='hide', font=dict(size=12, color='black'))


fig.show()



In [ ]:
fig = make_subplots(1, 6, specs=[[{'type':'domain'}, {'type':'domain'},
                                  {'type':'domain'}, {'type':'domain'},
                                  {'type':'domain'}, {'type':'domain'}]],
                    subplot_titles=["Iquique", "Valparaíso", "Santiago","Concepción","Valdivia","Puerto Montt"])
fig.add_trace(go.Pie(labels=f_2020_iquique_total.index, values=f_2020_iquique_total.values,scalegroup='one'), 1, 1)
fig.add_trace(go.Pie(labels=f_2020_valpo_total.index, values=f_2020_valpo_total.values, scalegroup='one'), 1, 2)
fig.add_trace(go.Pie(labels=f_2020_stgo_total.index, values=f_2020_stgo_total.values, scalegroup='one'), 1, 3)
fig.add_trace(go.Pie(labels=f_2020_conce_total.index, values=f_2020_conce_total.values, scalegroup='one'), 1, 4)
fig.add_trace(go.Pie(labels=f_2020_valdi_total.index, values=f_2020_valdi_total.values, scalegroup='one'), 1, 5)
fig.add_trace(go.Pie(labels=f_2020_puerto_total.index, values=f_2020_puerto_total.values, scalegroup='one'), 1, 6)
fig.update_traces(marker=dict(line=dict(color='#000000', width=1.5),colors=['#377eb8','#dede00','#999999','#f781bf','#ff7f00']),
                  textinfo='none'
                  #textfont_size=10, textinfo='percent',texttemplate='%{percent:.1%}'
                )
fig.update_layout(uniformtext_minsize=10, uniformtext_mode='hide', font=dict(size=12, color='black'))
fig.show()

# Network Analysis

In [ ]:
# Calcula la matriz de correlación de Pearson entre las comunas para el sector residencial
correlation_matrix = df.groupby('nom_com').apply(lambda x: x.pivot(index='Years', columns='Level_1', values='Value').corr().fillna(0))

# Imprimir las etiquetas únicas de las comunas utilizadas en la matriz de correlación
unique_labels = correlation_matrix.index.unique()
print(unique_labels)


In [ ]:
# Genera la matriz de correlación de Pearson entre todas las comunas para el sector residencial
pivot_df = df.pivot_table(index='Years', columns=['nom_com', 'Level_1'], values='Value', aggfunc='mean').fillna(0)
correlation_matrix = pivot_df.corr().fillna(0)

# Define un umbral de correlación
umbral_correlacion = 0.7

# Filtra únicamente los nombres de comunas para nodos
comunas = df['nom_com'].unique()

# Crea un grafo vacío
G = nx.Graph()

# Agrega nodos (comunas) al grafo
G.add_nodes_from(comunas)

# Agrega enlaces (conexiones) basados en la correlación por encima del umbral, solo entre nombres de comunas
for i, comuna1 in enumerate(comunas):
    for j, comuna2 in enumerate(comunas):
        correlation_value = correlation_matrix.loc[(comuna1, 'Energy'), (comuna2, 'Energy')]
        if i < j and correlation_value > umbral_correlacion:
            print(f"Correlación entre {comuna1} y {comuna2}: {correlation_value}")
            G.add_edge(comuna1, comuna2)

# Visualiza el grafo
nx.draw(G, with_labels=True)


In [ ]:
# Seleccionar una muestra de comunas (por ejemplo, las primeras 10)
muestra_comunas = df['nom_com'].unique()[:30]

# Filtrar el DataFrame original para incluir solo la muestra de comunas seleccionada
df_muestra = df[df['nom_com'].isin(muestra_comunas)]

# Generar la matriz de correlación de Pearson entre la muestra de comunas para el sector residencial
pivot_df_muestra = df_muestra.pivot_table(index='Years', columns=['nom_com', 'Level_1'], values='Value', aggfunc='mean').fillna(0)
correlation_matrix_muestra = pivot_df_muestra.corr().fillna(0)

# Definir un umbral de correlación
umbral_correlacion = 0.5

# Filtrar únicamente los nombres de comunas para nodos en la muestra
comunas_muestra = df_muestra['nom_com'].unique()

# Crear un grafo vacío
G_muestra = nx.Graph()

# Agregar nodos (comunas) al grafo para la muestra
G_muestra.add_nodes_from(comunas_muestra)

# Agregar enlaces (conexiones) basados en la correlación por encima del umbral, solo entre nombres de comunas en la muestra
for i, comuna1 in enumerate(comunas_muestra):
    for j, comuna2 in enumerate(comunas_muestra):
        correlation_value = correlation_matrix_muestra.loc[(comuna1, 'Energy'), (comuna2, 'Energy')]
        if i < j and correlation_value > umbral_correlacion:
            print(f"Correlación entre {comuna1} y {comuna2}: {correlation_value}")
            G_muestra.add_edge(comuna1, comuna2)

# Visualizar el grafo para la muestra de comunas
nx.draw(G_muestra, with_labels=True)


In [ ]:
# Comunas específicas que deseas mostrar en el grafo
comunas_especificas = ['PUERTO OCTAY', 'PUERTO MONTT', 'CHONCHI', 'QUELLÓN', 'OSORNO', 'ANCUD']

# Filtrar el DataFrame original para incluir solo las comunas específicas
df_especifico = df[df['nom_com'].isin(comunas_especificas)]

# Generar la matriz de correlación de Pearson entre las comunas específicas para el sector residencial
pivot_df_especifico = df_especifico.pivot_table(index='Years', columns=['nom_com', 'Level_1'], values='Value', aggfunc='mean').fillna(0)
correlation_matrix_especifico = pivot_df_especifico.corr().fillna(0)

# Definir un umbral de correlación
umbral_correlacion = 0.7

# Filtrar únicamente los nombres de comunas para nodos en las comunas específicas
comunas_especificas = df_especifico['nom_com'].unique()

# Crear un grafo vacío
G_especifico = nx.Graph()

# Agregar nodos (comunas) al grafo para las comunas específicas
G_especifico.add_nodes_from(comunas_especificas)

# Agregar enlaces (conexiones) basados en la correlación por encima del umbral, solo entre nombres de comunas en las comunas específicas
for i, comuna1 in enumerate(comunas_especificas):
    for j, comuna2 in enumerate(comunas_especificas):
        correlation_value = correlation_matrix_especifico.loc[(comuna1, 'Energy'), (comuna2, 'Energy')]
        if i < j and correlation_value > umbral_correlacion:
            print(f"Correlación entre {comuna1} y {comuna2}: {correlation_value}")
            G_especifico.add_edge(comuna1, comuna2)

# Visualizar el grafo para las comunas específicas
nx.draw(G_especifico, with_labels=True)


In [ ]:
# Código para generar la matriz de correlación, crear el grafo y agregar los enlaces (similar al código anterior)

# Crear una lista de nodos que están conectados (tienen enlaces) en el grafo
nodos_conectados = [nodo for nodo, grado in G.degree if grado > 0]

# Crear un nuevo grafo con solo los nodos que están conectados
G_nodos_conectados = G.subgraph(nodos_conectados)

# Visualizar el grafo con solo las comunas que tienen conexiones
nx.draw(G_nodos_conectados, with_labels=True)


In [ ]:
# Crear el layout para el grafo (spring_layout es un ejemplo, puedes usar otro)
#layout = nx.spring_layout(G_nodos_conectados)
layout = nx.circular_layout(G_nodos_conectados)


# Visualizar el grafo sin etiquetas
plt.figure(figsize=(10, 10))
nx.draw(G_nodos_conectados, pos=layout, with_labels=False, node_size=30, edge_color='black')
#nx.draw_circular(G_nodos_conectados,with_labels=False, node_size=60, edge_color='black')
plt.axis('off')
plt.show()


In [ ]:
# Crear el layout para el grafo (spring_layout es un ejemplo, puedes usar otro)
layout = nx.kamada_kawai_layout(G_nodos_conectados)

# Visualizar el grafo sin etiquetas
plt.figure(figsize=(10, 10))
nx.draw(G_nodos_conectados, pos=layout, with_labels=False, node_size=60, node_color='teal', edge_color='grey')
plt.axis('off')
plt.show()


In [ ]:
#DATO FREAK

#Kamada-Kawai Tomihisa Kamada, Satoru Kawai. An Algorithm for Drawing General Undirected Graphs. Information Processing Letters, 31:7-15, 1988.
#Kamada Kawai utiliza el método de Newton-Raphson para optimizar con respecto a un solo vértice. Al iterativamente realizar la solución para cada vértice se reduce el estrés en general.
#Mediante la aproximación y minimizar la ecuación de estrés (a suma cuadrada de las diferencias entre la distancia ideal y la distancia real para todos los vértices), el método de Kamada - Kawai conserva el equilibrio total de un grafo, y produce diseños con pequeñas cantidades de los cruces de enlace.

### Unir los NLWU DataFrames / Concat NLWU DataFrame

In [ ]:
# Lista de archivos CSV
files_tot_1 = ['DrinkingWater\CR2WU_NLU_DrinkingWater_historical_1950-2020_communes.csv',
                'Energy\CR2WU_NLU_Energy_historical_1950-2020_communes.csv',
                'Industry\CR2WU_NLU_Industry_historical_1950-2020_communes.csv',
                'Livestock\CR2WU_NLU_Livestock_historical_1950-2020_communes.csv',
                'Mining\CR2WU_NLU_Mining_historical_1950-2020_communes.csv']

# DataFrame vacío para el nuevo formato
df_nuevo = pd.DataFrame(columns=['Date', 'Commune', 'Value', 'Level_1'])

# For que va leyendo la lista / Reading list loop
for i in files_tot_1:
    file_path = os.path.join(subfolder_path_csv, i)  # Establecer ruta donde está el DataFrame / Set DataFrame path
    NLU_i_communes = pd.read_csv(file_path)  # Leer archivo DataFrame (los de la lista) / Read DataFrame files (from the list)
    level_1_value = i.split('_')[-4]  # Obtener anteantepenúltima palabra del path, que es el valor del nivel 1 (DrinkingWater, Energy...) / Obtain level 1 value

    # Iterar a través de las columnas de comuna y sus valores
    for commune_column in NLU_i_communes.columns[1:]:
        df_temp = pd.DataFrame()  # DataFrame temporal para cada iteración
        df_temp['Date'] = NLU_i_communes['Date']
        df_temp['Commune'] = commune_column
        df_temp['Value'] = NLU_i_communes[commune_column]
        df_temp['Level_1'] = level_1_value  # Reemplaza 'DrinkingWater' por el nivel correspondiente

        # Concatenar el DataFrame temporal al DataFrame total
        df_nuevo = pd.concat([df_nuevo, df_temp], ignore_index=True)

# Muestra el nuevo DataFrame
print(df_nuevo)


In [ ]:
df_nuevo.head()

#### Verificar si se leyero y guardaron bien los dataframes

In [ ]:
# Leerlos y guardarlos por separado
NLU_Energy_communes = pd.read_csv(subfolder_path_csv + r'\Energy\CR2WU_NLU_Energy_historical_1950-2020_communes.csv')
NLU_Mining_communes = pd.read_csv(subfolder_path_csv + r'\Mining\CR2WU_NLU_Mining_historical_1950-2020_communes.csv')
NLU_Livestock_communes = pd.read_csv(subfolder_path_csv + r'\Livestock\CR2WU_NLU_Livestock_historical_1950-2020_communes.csv')

In [ ]:
# Verificar que hayan valores distintos de cero 
any_non_zero = np.any(NLU_Mining_communes != 0)

if any_non_zero:
    # Encontrar los índices y columnas donde se encuentran los valores distintos de cero
    indices, columnas = np.where(NLU_Mining_communes != 0)

    for indice, columna in zip(indices, columnas):
        valor = NLU_Mining_communes.iloc[indice, columna]
        print(f"Valor distinto de cero encontrado en el índice {indice} de la columna {NLU_Mining_communes.columns[columna]}: {valor}")
else:
    print("No se encontraron valores distintos de cero en el DataFrame.")



In [ ]:
print(NLU_Mining_communes.query("Date == '1951-01-01'")['4101'])
print(df_nuevo.query("Date == '1951-01-01' & Commune == '4101' & Level_1 == 'Mining'"))

In [ ]:
#Algunas funciones útiles

#df_nuevo.describe()
#NLU_communes.columns #muestra las columnas del dataframe
#NLU_communes.columns.size #cuántas columnas son = 344. Osea 343 comunas + columna Date
#NLU_communes.shape #muestra el tamaño del dataframe 71 años 344 columnas (date + 3434 comunas)
#df_nuevo['Commune'].size #total de filas 24353, está bien porque son 343comunas x 71 años
#df_nuevo['Commune'].unique().size #343 comunas 
#print(df_nuevo['Level_1'].unique())
#print(NLU_Energy_communes.iloc[70])
#NLU_Energy_communes['Date'].dtype #para ver qué tipo de dato es. Si es dtype('O'), entonces es un objeto y hay que pasarlo a entero NLU_Energy_communes['Date'].astype(int)

### Limpieza NLWU data

In [ ]:
# Fecha a año / Date to year
df_nuevo['Date'] = pd.to_datetime(df_nuevo["Date"])
df_nuevo['Date'] = df_nuevo['Date'].dt.strftime('%Y')
df_nuevo = df_nuevo.rename(columns={"Date": "Years"})
df_nuevo['Years'] = df_nuevo['Years'].astype(int) #para que sea un número y no un objeto
df_nuevo.head()

In [ ]:
# Nombre y código de comunas y regiones 
df_nuevo['Commune'] = df_nuevo['Commune'].astype(int) #para que sea un número y no un objeto
df_nuevo = df_nuevo.rename(columns={"Commune":"cod_com"}) #cambiarle el nombre a la columna
df_nuevo = df_nuevo.merge(cod[['cod_com','nom_com','cod_reg','nom_reg']], on='cod_com', how='left')
df_nuevo.head()
# Ahora df_nuevo contiene las columnas 'Years', 'Value', 'Commune_cod', 'Commune_name', 'Region_cod', y 'Region_name'

In [ ]:
# Forma de asociar valores en un diccionario (al final lo hice directo con el dataframe)
# #df_nuevo = df_nuevo.rename(columns={"Commune":"Commune_cod"})
# df_nuevo_prueba = df_nuevo
# # Crear una nueva columna 'Commune_cod' basada en 'Commune'
# df_nuevo_prueba['Commune_cod'] = df_nuevo_prueba['Commune']

# # Utilizar map para asignar los nombres de las comunas basados en el código
# df_nuevo_prueba['Commune_name'] = df_nuevo_prueba['Commune_cod'].map(cod_com_nom_com)

# df_nuevo_prueba.head()

#### Limpieza de datos antiguo

In [ ]:
#Database cleaning
NLU_nation['Date'] = pd.to_datetime(NLU_nation["Date"])
NLU_nation['Date'] = NLU_nation['Date'].dt.strftime('%Y')
NLU_nation = NLU_nation.rename(columns={"Date": "Years","1":"NLU"})
NLU_nation['Years'] = NLU_nation['Years'].astype(int)

NLU_regions['Date'] = pd.to_datetime(NLU_regions["Date"])
NLU_regions['Date'] = NLU_regions['Date'].dt.strftime('%Y')
NLU_regions = NLU_regions.rename(columns={"Date": "Years"})
NLU_regions['Years'] = NLU_regions['Years'].astype(int)

#Adding Region's names
NLU_regions_plot = NLU_regions.rename(columns=cod_reg_nom_reg)

#NLU national 2 decimal
NLU_nation_ = NLU_nation
NLU_nation_['NLU'] = NLU_nation_['NLU'].round(decimals=2)

#Database cleaning communes
NLU_communes['Date'] = pd.to_datetime(NLU_communes["Date"])
NLU_communes['Date'] = NLU_communes['Date'].dt.strftime('%Y')
NLU_communes = NLU_communes.rename(columns={"Date": "Years"})
NLU_communes ['Years'] = NLU_communes['Years'].astype(int)

#Adding Region's names
NLU_communes_plot = NLU_communes.rename(columns=cod_com_nom_com)


In [ ]:
cod_com_nom_com

In [ ]:
#Database cleaning communes

NLU_DW_communes['Date'] = pd.to_datetime(NLU_DW_communes["Date"])
NLU_DW_communes['Date'] = NLU_DW_communes['Date'].dt.strftime('%Y')
NLU_DW_communes = NLU_DW_communes.rename(columns={"Date": "Years"})
NLU_DW_communes ['Years'] = NLU_DW_communes['Years'].astype(int)
NLU_DW_communes_plot = NLU_DW_communes.rename(columns=cod_com_nom_com)

NLU_Energy_communes['Date'] = pd.to_datetime(NLU_Energy_communes["Date"])
NLU_Energy_communes['Date'] = NLU_Energy_communes['Date'].dt.strftime('%Y')
NLU_Energy_communes = NLU_Energy_communes.rename(columns={"Date": "Years"})
NLU_Energy_communes ['Years'] = NLU_Energy_communes['Years'].astype(int)
NLU_Energy_communes_plot = NLU_Energy_communes.rename(columns=cod_com_nom_com)

NLU_Industry_communes['Date'] = pd.to_datetime(NLU_Industry_communes["Date"])
NLU_Industry_communes['Date'] = NLU_Industry_communes['Date'].dt.strftime('%Y')
NLU_Industry_communes = NLU_Industry_communes.rename(columns={"Date": "Years"})
NLU_Industry_communes ['Years'] = NLU_Industry_communes['Years'].astype(int)
NLU_Industry_communes_plot = NLU_Industry_communes.rename(columns=cod_com_nom_com)

NLU_Livestock_communes['Date'] = pd.to_datetime(NLU_Livestock_communes["Date"])
NLU_Livestock_communes['Date'] = NLU_Livestock_communes['Date'].dt.strftime('%Y')
NLU_Livestock_communes = NLU_Livestock_communes.rename(columns={"Date": "Years"})
NLU_Livestock_communes ['Years'] = NLU_Livestock_communes['Years'].astype(int)
NLU_Livestock_communes_plot = NLU_Livestock_communes.rename(columns=cod_com_nom_com)

NLU_Mining_communes['Date'] = pd.to_datetime(NLU_Mining_communes["Date"])
NLU_Mining_communes['Date'] = NLU_Mining_communes['Date'].dt.strftime('%Y')
NLU_Mining_communes = NLU_Mining_communes.rename(columns={"Date": "Years"})
NLU_Mining_communes ['Years'] = NLU_Mining_communes['Years'].astype(int)
NLU_Mining_communes_plot = NLU_Mining_communes.rename(columns=cod_com_nom_com)

## Data summary

In [ ]:
#Mean Region
# # Calcular el valor promedio de todo el dataframe (desde la columna 1 en adelante)
# mean_total = np.mean(NLU_regions_plot.iloc[:, 1:].values)
# # Calcular la desviación estándar de todo el dataframe (desde la columna 1 en adelante)
# std_total = np.std(NLU_regions_plot.iloc[:, 1:].values)
# print("\nConsumo promedio regional:")
# print(mean_total)
# print("\nDesviación estándar:")
# print(std_total)

#Mean Comuna
# Calcular el valor promedio de todo el dataframe (desde la columna 1 en adelante)
cmean_total = np.mean(NLU_communes_plot.iloc[:, 1:].values)
cmean_total_y = (cmean_total* 3600 * 24 * 365)/1e+9
# Calcular la desviación estándar de todo el dataframe (desde la columna 1 en adelante)
cstd_total = np.std(NLU_communes_plot.iloc[:, 1:].values)
cstd_total_y = (cstd_total* 3600 * 24 * 365)/1e+9
print("\nConsumo promedio comunal:", cmean_total, '[m^3/s]',cmean_total_y, '[km^3/y]')
print("\nDesviación estándar total:", cstd_total, '[m^3/s]',cstd_total_y, '[km^3/y]')

In [ ]:
#Min and max Región

# Encontrar el valor máximo y mínimo y su columna en cada fila
max_values = NLU_regions_plot.iloc[:, 1:].max(axis=1)  # Obtener el valor máximo en cada fila
max_columns = NLU_regions_plot.iloc[:, 1:].idxmax(axis=1)  # Obtener la columna donde está el valor máximo en cada fila
min_values = NLU_regions_plot.iloc[:, 1:].min(axis=1)  # Obtener el valor mínimo en cada fila
min_columns = NLU_regions_plot.iloc[:, 1:].idxmin(axis=1)  # Obtener la columna donde está el valor mínimo en cada fila


# Encontrar el índice donde se encuentra el valor máximo y mínimo  global
max_global_index = max_values.idxmax()  # Índice donde se encuentra el valor máximo global
max_global_year = NLU_regions_plot.loc[max_values.idxmax(),'Years']  # Año donde se encuentra el valor máximo global
max_global_column = max_columns[max_global_index]  # Columna donde está el valor máximo global
max_global_value = max_values[max_global_index]  # Valor máximo global
max_global_value_y = (max_global_value* 3600 * 24 * 365)/1e+9 # Valor máximo global [km3/y]
min_global_index = min_values.idxmin()  # Índice donde se encuentra el valor máximo global
min_global_year = NLU_regions_plot.loc[min_values.idxmin(),'Years']  # Año donde se encuentra el valor máximo global
min_global_column = min_columns[min_global_index]  # Columna donde está el valor máximo global
min_global_value = min_values[min_global_index]  # Valor máximo global
min_global_value_y = (min_global_value* 3600 * 24 * 365)/1e+9 # Valor máximo global [km3/y]


print("\nRegión del máximo global:", max_global_column)
print("\nAño valor máximo global:", max_global_year)
print("Valor máximo global:", max_global_value, '[m^3/s]',max_global_value_y, '[km^3/y]')

print("\nRegión del mínimo global:", min_global_column)
print("\nAño valor mínimo global:", min_global_year)
print("Valor mínimo global:", min_global_value, '[m^3/s]',min_global_value_y, '[km^3/y]')


In [ ]:
#Min and max Comuna

# Encontrar el valor máximo y mínimo y su columna en cada fila
cmax_values = NLU_communes_plot.iloc[:, 1:].max(axis=1)  # Obtener el valor máximo en cada fila
cmax_columns = NLU_communes_plot.iloc[:, 1:].idxmax(axis=1)  # Obtener la columna donde está el valor máximo en cada fila
cmin_values = NLU_communes_plot.iloc[:, 1:].min(axis=1)  # Obtener el valor máximo en cada fila
cmin_columns = NLU_communes_plot.iloc[:, 1:].idxmin(axis=1)  # Obtener la columna donde está el valor máximo en cada fila


# Encontrar el índice donde se encuentra el valor máximo y mínimo  global
cmax_global_index = cmax_values.idxmax()  # Índice donde se encuentra el valor máximo global
cmax_global_year = NLU_communes_plot.loc[cmax_values.idxmax(),'Years']  # Año donde se encuentra el valor máximo global
cmax_global_column = cmax_columns[cmax_global_index]  # Columna donde está el valor máximo global
cmax_global_value = cmax_values[cmax_global_index]  # Valor máximo global
cmax_global_value_y = (cmax_global_value* 3600 * 24 * 365)/1e+9 # Valor máximo global [km3/y]
cmin_global_index = cmin_values.idxmin()  # Índice donde se encuentra el valor máximo global
cmin_global_year = NLU_communes_plot.loc[cmin_values.idxmin(),'Years']  # Año donde se encuentra el valor máximo global
cmin_global_column = cmin_columns[cmin_global_index]  # Columna donde está el valor máximo global
cmin_global_value = cmin_values[cmin_global_index]  # Valor mínimo global
cmin_global_value_y = (cmin_values[cmin_global_index]* 3600 * 24 * 365)/1e+9  # Valor mínimo global [km3/y]


print("\nComuna del máximo global:", cmax_global_column)
print("\nAño valor máximo global:", cmax_global_year)
print("Valor máximo global:", cmax_global_value, '[m^3/s]',cmax_global_value_y, '[km^3/y]')


print("\nComuna del mínimo global:", cmin_global_column)
print("\nAño valor mínimo global:", cmin_global_year)
print("Valor mínimo global:", cmin_global_value, '[m^3/s]',cmin_global_value_y, '[km^3/y]')


In [ ]:
cmax_global_year* 3600 * 24 * 365

In [ ]:
import os

In [ ]:
NLU_DW_communes_plot.head()

In [ ]:
display(NLU_Mining_communes_plot.sample(10))

In [ ]:
#Useful lists
year_list = NLU_regions['Years'].tolist()
reg_list = [i for i in cod_reg_nom_reg.values()] #not ordered by lat
com_list = [i for i in cod_com_nom_com.values()] #not ordered by lat

In [ ]:
display(NLU_communes.sample(10))

In [ ]:
#Adding yearly use
#HACER UNA FUNCIÓN DE ESTO


#Def days per year
years_with_365_days = []
years_with_366_days = []

for year in range(1950, 2021):
    date = datetime.date(year, 12, 31)  # Crear una fecha para el último día del año
    
    if date.timetuple().tm_yday == 365:
        years_with_365_days.append(year)

    if date.timetuple().tm_yday == 366:
        years_with_366_days.append(year)

y_365=[str(item) for item in years_with_365_days]
y_366=[str(item) for item in years_with_366_days]

#Yearly water use function

def calcular_consumo_anual(row):
    year_col = 'Years'  # Nombre de la columna que contiene los años en el dataframe
    use_col = 'NLU'  # Nombre de la columna que contiene el consumo diario en el dataframe

    year = row[year_col]  # Obtener el valor del año
    use_value = row[use_col]  # Obtener el valor del consumo diario

    if str(year) in y_365:
        return use_value * 3600 * 24 * 365  # 1/s * 3600s/hr * 24hr/d * 365d/year
    elif str(year) in y_366:
        return use_value * 3600 * 24 * 366  # 1/s * 3600s/hr * 24hr/d * 366d/year
    else:
        return None  # Otra opción si el año no está en la lista de 365 días
    
NLU_nation['NLU_year'] = NLU_nation.apply(calcular_consumo_anual, axis=1)/1e+9
NLU_nation['NLU_year'] = NLU_nation['NLU_year'].astype(float).round(2)


### Database reading function

In [ ]:
def get_smallest_unit_file(path):
    # Get all files in the directory
    files = os.listdir(path)
    
    # Filter out directories and sort to get consistent order
    files = sorted(file for file in files if os.path.isfile(os.path.join(path, file)))
    
    # Return the last file (smallest unit file) or None if no files found
    return files[-1] if files else None

In [ ]:
get_smallest_unit_file(r'C:\Users\Mariana\Google Drive\Doctorado\Doctorado_UACh\Tesis\BBDD\CR2\NLU\csv\Total')

In [ ]:
def list_files(input_directory):
    lista_ultimos_archivos = []
    for root, _, _ in os.walk(input_directory):
        # Get the file in the current directory (smallest unit file)
        smallest_unit_file = get_smallest_unit_file(root)
        if smallest_unit_file:
            file_path = os.path.join(root, smallest_unit_file)
            print(file_path)
            lista_ultimos_archivos = lista_ultimos_archivos + [file_path]
            

    return lista_ultimos_archivos


In [ ]:
def concatenate_files(files_list, output_dir):
    for index, file_path in enumerate(files_list): 
        file = pd.read_csv(file_path) # acá hay que ver cómo son los separadores y esas cosas para que lo lea bien 
        file['fuente'] = os.path.dirname(file_path).replace('/', '_')
        if index == 0:
            concatenated_file = file
        else: 
            concatenated_file = pd.concat([concatenated_file, file])
    # aquí en vez de retornarlo lo podemos escribir en csv
    concatenated_file.to_csv(output_dir)
    return concatenated_file 


In [ ]:
unidades_minimas[0]

In [ ]:
unidades_minimas = list_files(r'C:\Users\Mariana\Google Drive\Doctorado\Doctorado_UACh\Tesis\BBDD\CR2\NLU\csv\Total')
print(unidades_minimas)
#archivo_consolidado = concatenate_files(unidades_minimas, 'output.csv')

In [ ]:
#National data
total_use = NLU_nation['NLU'].sum()
total_use_km3 = NLU_nation['NLU_year'].sum()

#Min and max
year_max = NLU_nation.loc[NLU_nation['NLU'].idxmax(), 'Years']
year_min = NLU_nation.loc[NLU_nation['NLU'].idxmin(), 'Years']
use_max = NLU_nation['NLU'].max()
use_min = NLU_nation['NLU'].min()
use_max_km3 = NLU_nation['NLU_year'].max()
use_min_km3 = NLU_nation['NLU_year'].min()


#Mean use in the last 10 years
last_years = (NLU_nation['Years'] >= 2010) & (NLU_nation['Years'] <= 2020)
NLU_last_years = NLU_nation[last_years]
mean_use = NLU_last_years['NLU'].mean()

#Mean use since 2002
years_2002 = (NLU_nation['Years'] >= 2002) & (NLU_nation['Years'] <= 2020)
NLU_2002 = NLU_nation[years_2002]
mean_use_2002 = NLU_2002['NLU'].mean()
mean_use_2002_km3 = NLU_2002['NLU_year'].mean()


print("Total water use:", f"{total_use:.0f}""[m³/s]","or", f"{total_use_km3:.0f}""[km³/y]")
print("Highest water use year:", year_max, "with", f"{use_max:.0f}" "[m³/s]","or", f"{use_max_km3:.0f}","[km³/y]")
print("Lowest water use year:", year_min, "with", f"{use_min:.0f}" "[m³/s]","or", f"{use_min_km3:.0f}","[km³/y]")
print("Mean water during the last 10 years:", f"{mean_use:.0f}""[m³/s]","or",  f"{mean_use_2002_km3:.0f}""[km³/y]")


In [ ]:
#Según el Informe Mundial de las Naciones Unidas sobre el Desarrollo de los Recursos Hídricos, 
#el consumo total de agua dulce a nivel mundial se estima en alrededor de 4.600 kilómetros cúbicos (km³) por año.
#y según Naciones Unidas, existen 195 países

#4600/195 = 23.6 => cada país usa 23.6[km3] de agua dulce al año
#92.2/23.6 =3.9 => pero Chile, usa casi 4 veces ese volumen (92.2[km3] usados en promedio entre 2010 y 2020) 

In [ ]:
#Graph of yearly national (Non Land) Water Use [km3]

# Ajustar el modelo de regresión lineal
model = sm.OLS(NLU_nation['NLU'], sm.add_constant(NLU_nation['Years']))
results = model.fit()
r_squared = results.rsquared

# English
fig = px.scatter(NLU_nation, x="Years", y="NLU",
                 title="Non Land Water Use in Chile",
                 trendline="ols",
                 labels={'Years': 'Years', 'NLU': 'Water use [m³/s]'})
fig.add_annotation(
    x=2020, y=1000,
    text=f"R-squared: {r_squared:.2f}",
    showarrow=False,
    font=dict(size=12),
    align="right")

fig.show()


#Spanish
#fig2 = px.line(NLU_nation, x="Years", y="NLU_year", 
fig2 = px.scatter(NLU_nation, x="Years", y="NLU", 
               title="Uso de agua (no relacionado con la tierra) de Chile", 
               trendline="ols",
               labels={'Years':'Años', 'NLU':'Uso de agua [m³/s]'})
fig2.add_annotation(
    x=2020, y=1000,
    text=f"R cuadrado: {r_squared:.2f}",
    showarrow=False,
    font=dict(size=12),
    align="right")

fig2.show()


In [ ]:
#Graph of yearly national (Non Land) Water Use [km3] lasts years. Non linear adj.
 
# English
fig5 = px.scatter(NLU_last_years, x="Years", y="NLU", color_discrete_sequence=['purple'],
                 title="Non Land Water Use in Chile since 2010",
                 trendline="lowess", trendline_options=dict(frac=0.1),
                 labels={'Years': 'Years', 'NLU': 'Water use [m³/s]'})

fig5.show()


#Spanish
fig6 = px.scatter(NLU_last_years, x="Years", y="NLU", color_discrete_sequence=['purple'],
               title="Uso de agua (no relacionado con la tierra) de Chile desde el 2010", 
               trendline="lowess", trendline_options=dict(frac=0.1),
               labels={'Years':'Años', 'NLU':'Uso de agua [m³/s]'})


fig6.show()


In [ ]:
#Graph of yearly national (Non Land) Water Use years. Moving Average
 
#English
fig1 = px.scatter(NLU_nation, x="Years", y="NLU",
                 labels={'Years': 'Years', 'NLU': 'Water use [m³/s]'},
                 title="5-point moving average",
                 trendline="rolling", trendline_options=dict(window=5))
fig2 = px.scatter(NLU_nation, x="Years", y="NLU", color_discrete_sequence=['green'],
                  labels={'Years': 'Years', 'NLU': 'Water use [m³/s]'},
                  title="10-point moving average",
                  trendline="rolling", trendline_options=dict(window=10))
fig1.show()
fig2.show()

#Español
fig3 = px.scatter(NLU_nation, x="Years", y="NLU",
                 labels={'Years':'Años', 'NLU':'Uso de agua [m³/s]'},
                 title="Promedio móvil de 5 puntos",
                 trendline="rolling", trendline_options=dict(window=5))
fig4 = px.scatter(NLU_nation, x="Years", y="NLU", color_discrete_sequence=['green'],
                  labels={'Years':'Años', 'NLU':'Uso de agua [m³/s]'},
                  title="Promedio móvil de 10 puntos",
                  trendline="rolling", trendline_options=dict(window=10))
fig3.show()
fig4.show()

# Moving_average_5 = NLU_nation.rolling(window=5).mean()
# Moving_average_10 = NLU_nation.rolling(window=10).mean()

In [ ]:
#Graph of yearly national (Non Land) Water Use [km3] since 2002. Non linear adj.
 
# English
fig7 = px.scatter(NLU_2002, x="Years", y="NLU", color_discrete_sequence=['orange'],
                 title="Non Land Water Use in Chile since 2002",
                 #trendline="lowess", trendline_options=dict(frac=0.1),
                 trendline="ols",
                 labels={'Years': 'Years', 'NLU': 'Water use [m³/s]'})

fig7.show()


#Spanish
fig8 = px.scatter(NLU_2002, x="Years", y="NLU", color_discrete_sequence=['orange'],
               title="Uso de agua (no relacionado con la tierra) de Chile desde el 2002", 
               trendline="lowess", trendline_options=dict(frac=0.1),
               labels={'Years':'Años', 'NLU':'Uso de agua [m³/s]'})


fig8.show()

In [ ]:
#llevar esto a packages

from statsmodels.nonparametric.smoothers_lowess import lowess 

In [ ]:
# Ajustar el modelo de regresión lineal para todos los datos
model_all = sm.OLS(NLU_nation['NLU'], sm.add_constant(NLU_nation['Years']))
results_all = model_all.fit()

# Obtener el valor R cuadrado y la pendiente para todos los datos
r_squared_all = results_all.rsquared
slope_all = results_all.params['Years']

# Filtrar los datos para obtener solo los años mayores a 2002
filtered_data = NLU_nation[NLU_nation['Years'] >= 2002]

# Calcular los residuos utilizando Lowess
residuals = filtered_data['NLU'] - lowess(filtered_data['NLU'], filtered_data['Years'], frac=0.5)[:, 1]
# Agregar los residuos como una columna adicional en el DataFrame
filtered_data.loc[:, 'Residuals'] = residuals

# Crear el gráfico de dispersión con los datos completos
fig = px.scatter(NLU_nation, x="Years", y="NLU",
                 trendline="ols",
                 color_discrete_sequence=['blue'],
                 labels={'Years': 'Years', 'NLU': 'Water use [km³]'})

# Agregar anotación con el valor R cuadrado y la pendiente para los datos completos
fig.add_annotation(
    x=2020, y=1000,
    text=f"R-squared (All): {r_squared_all:.2f}\nSlope (All): {slope_all:.2f}",
    showarrow=False,
    font=dict(size=12),
    align="left"
)

# Agregar el trazo para los datos filtrados en un color distinto
fig.add_traces(px.scatter(filtered_data, x="Years", y="NLU", 
                          #trendline="ols", 
                          trendline="lowess", 
                          hover_data=['Residuals'],
                          color_discrete_sequence=['red']).data)

# # Agregar anotación con el valor R cuadrado y la pendiente para los datos filtrados
# fig.add_annotation(
#     x=2020, y=1200,
#     text=f"R-squared (Filtered): {r_squared_filtered:.2f}\nSlope (Filtered): {slope_filtered:.2f}",
#     showarrow=False,
#     font=dict(size=12),
#     align="right"
# )

fig.show()


In [ ]:
#Graph of yearly national (Non Land) Water Use [m3/s]

#English
fig = px.scatter(NLU_nation, x="Years", y="NLU", trendline='ols',
              title="Non Land Water Use in Chile", 
              labels={'Years':'Years', 'NLU':'Water use [m³/s]'}) 
fig.show()

#Spanish
fig2 = px.line(NLU_nation, x="Years", y="NLU", 
               title="Uso de agua no relacionado con la tierra", 
               labels={'Years':'Años', 'NLU':'Uso de agua [m³/s]'})
fig2.show()

Reu con Mauro.
Plotear log uso agua -no solo log hab-
Agregar número de habitantes antes del 2000, por crecimiento de la población
Agregar variables ambientales, fenómeno del Niño ENSO con correlaciones cruzadas... y luego hablar con Horacio por si hay otro estadístico, o precipitación anual ¿
Papers de Carmen Paz Silva. índice socioeconómico Casen. Intentaron poner el PIB, pero se les complicó. Otra variable que utilizaron como proxy de paisaje urbano el nivel de sellamiento de superficie.
cap 1 caracterizar el consumo hídrico, abastecimiento y factores que influyen en él: regímen de lluvia, efecto ENSO sobre el consumo per cápita.
cap 2 dinámica de tipo de consumo. 
cap 3 consumo de agua domiciliario.
cap 4 semi cuantitativo, de gestión. análisis de situación global, incluir riesgos, planificación importación y exportación.
Estandarizar curva por población, más que hacer relación entre consumo y población. Uso de agua per cápita


### National Water Use and Inhabitants

In [ ]:
ls

In [ ]:
pop = pd.read_csv(r'/mnt/home2/mari/Tesis/INE_2002-2035.csv', index_col=0, sep=';', encoding='latin-1')

In [ ]:
pop.head()

In [ ]:
#Refining Dataframe Population

#Eliminate columns
col_drop = ['Sexo (1=Hombre 2=Mujer)', 'Grupo edad']
pop = pop.drop(col_drop,axis=1)

#Changing column names
pop_grouped = pop.groupby(['Nombre Comuna', 'Area (1=Urbano 2=Rural)'])[pop.columns[6:39]].sum().reset_index()
pop = pop.rename(columns=lambda x: x.replace('Poblacion ', '') if x.startswith('Poblacion ') else x) #year columns
pop = pop.rename(columns=lambda x: x.replace('Nombre ', 'nom_') if x.startswith('Nombre ') else x) 
pop = pop.rename(columns={'Provincia':'cod_provincia'})
pop = pop.rename(columns=lambda x: x.replace('Region', 'region') if 'Region' in x else x)
pop = pop.rename(columns=lambda x: x.replace('Provincia', 'provincia') if 'Provincia' in x else x)
pop = pop.rename(columns=lambda x: x.replace('Comuna', 'comuna') if 'Comuna' in x else x)
pop = pop.rename(columns={'Area (1=Urbano 2=Rural)':'urban1_rural2'})
pop.index.name = 'region'
pop.head()

pop.shape

In [ ]:
#Tengo que hacerlo para pop grouped, lo borré sin querer
#Population and NL Water Use Dataframe
pop_tot = pop.iloc[:, 6:25].sum().astype(int) #total population
#pop_tot = pop.iloc[:, 6:25].shift().sum().astype(int)
pop_years = pd.DataFrame({'Years': pop_tot.index, 'Population': pop_tot.values})
pop_years.head()

In [ ]:
pop_years.shape

In [ ]:
NLU_pop = NLU_nation_[NLU_nation_['Years'] >= 2002].reset_index(drop=True) #using NLU_nation_ 2 decimals NLU
#NLU_pop = NLU_nation_[NLU_nation_['Years'] >= 2002].reset_index(drop=True) #using NLU_nation_ 2 decimals NLU
#NLU_pop['Population']=pop_years_g['Population'] #usar este cuando corrija lo de arriba (crear pop years grouped)
NLU_pop['Population']=pop_years['Population']
NLU_pop.head()

In [ ]:
fig = px.scatter(NLU_pop, x="Population", y="NLU", trendline = 'ols',
              title="Population and Water Use in Chile", 
              labels={'Population':'Population', 'NLU':'Water use [km³]'}) 



fig2 = go.Figure(data=go.Scatter(x=NLU_pop["Population"], y=NLU_pop["NLU"]))
fig2.update_traces(line=dict(color='red'))

fig.show()
fig2.show()

In [ ]:

import numpy as np
NLU_pop['Log_Pop']=np.log(NLU_pop['Population'])
NLU_pop['Log_NLU']=np.log(NLU_pop['NLU'])
fig = px.line(NLU_pop, x="Log_Pop", y="Log_NLU", 
              title="Population and Water Use in Chile", 
              labels={'Log_Pop':'Log Population', 'NLU':'Log Water use'}) 

fig.show()
fig.write_image(r"/mnt/home2/mari/Tesis/logpopul_logwater.png")

In [ ]:
pip install -U kaleido

In [ ]:
fig.write_image(r"/mnt/home2/mari/Tesis/popul_water.png")

In [ ]:
pop_tot.plot()

### Regional Water Use

In [ ]:
NLU_regions_plot.head()
fig = px.box(NLU_regions_plot, x=NLU_regions_plot.columns[1:], y="Years")

In [ ]:
NLU_regions_plot.columns[1:]


# Intento de clusterización

In [ ]:
df = pd.read_parquet('/home/mari/Documents/GitHub/tesis/CR2/bbdd_cr2_01.parquet')

In [ ]:
import matplotlib.pyplot as plt

# Supongamos que tienes un DataFrame llamado 'df' con tus datos

# Crear el boxplot de la columna 'Value' mostrando los cuartiles
plt.figure(figsize=(8, 6))
plt.boxplot(df['Value'], showfliers=False, showmeans=True)
plt.title('Boxplot de la columna Value con Cuartiles')
plt.ylabel('Valor')
plt.show()

# Obtener estadísticas resumidas, incluyendo los cuartiles
summary_stats = df['Value'].describe()

# Imprimir los cuartiles
print("Q1 (25%):", summary_stats['25%'])
print("Mediana (50%):", summary_stats['50%'])
print("Q3 (75%):", summary_stats['75%'])


In [ ]:
#Con esto se crean categorias nuevas basadas en los cuartiles
df['clase_consumo'] = pd.cut(df['Value'], bins=4, labels=['low', 'low_med', 'high_med', 'high'])
df.head(5)

#hacerlo para el promedio historico

In [ ]:
prom_com =df.groupby('nom_com').mean('Value')
prom_com =prom_com.drop(['Years', 'categ', 'Level_1_encoded', 'Cluster'], axis =1)

prom_com['clase_consumo'] = pd.cut(prom_com['Value'], bins=4, labels=['low', 'low_med', 'high_med', 'high'])
prom_com.head(5)

In [ ]:
plt.hist(clase_consumo, bins=number of bins)

In [ ]:
fig = px.box(NLU_regions_plot, y=NLU_regions_plot.columns[1:],
             labels={"variable": ""}
                     #;"value":"WU [m^3/s]"}
                      )
fig.update_layout(
    yaxis_title='WU [m^3/s]'
)

fig.show()

In [ ]:
NLU_regions_plot.head()

In [ ]:
fig = px.box(NLU_regions_plot, y=NLU_regions_plot.columns[group1_columns],
             labels={"variable": ""}
                     #;"value":"WU [m^3/s]"}
                      )
fig.update_layout(
    yaxis_title='WU [m^3/s]'
)

fig.add_annotation(x=2, y=5,
            text="Text annotation with arrow",
            showarrow=True,
            arrowhead=1)

year_max = NLU_nation.loc[NLU_nation['NLU'].idxmax(), 'Years']
year_min = NLU_nation.loc[NLU_nation['NLU'].idxmin(), 'Years']
use_max = NLU_nation['NLU'].max()
use_min = NLU_nation['NLU'].min()

fig.show()

In [ ]:
group1_columns = [6, 7, 8, 10]
group2_columns = [5, 13, 14]
group3_columns = [2, 4, 9, 16]
group4_columns = [3, 4, 11]
group5_columns = [1, 12, 15]


fig = px.box(NLU_regions_plot, y=NLU_regions_plot.columns[group1_columns],
             labels={"variable": ""}
                     #;"value":"WU [m^3/s]"}
                      )
fig.update_layout(
    yaxis_title='WU [m^3/s]'
)

fig.show()


fig2 = px.box(NLU_regions_plot, y=NLU_regions_plot.columns[group2_columns],
             labels={"variable": ""}
                     #;"value":"WU [m^3/s]"}
                      )
fig2.update_layout(
    yaxis_title='WU [m^3/s]'
)

fig2.show()


fig3 = px.box(NLU_regions_plot, y=NLU_regions_plot.columns[group3_columns],
             labels={"variable": ""}
                     #;"value":"WU [m^3/s]"}
                      )
fig3.update_layout(
    yaxis_title='WU [m^3/s]'
)
fig3.update_traces(quartilemethod="exclusive")

fig3.show()

fig4 = px.box(NLU_regions_plot, y=NLU_regions_plot.columns[group4_columns],
             labels={"variable": ""}
                     #;"value":"WU [m^3/s]"}
                      )
fig4.update_layout(
    yaxis_title='WU [m^3/s]'
)
fig4.update_traces(quartilemethod="exclusive")
fig4.show()

fig5 = px.box(NLU_regions_plot, y=NLU_regions_plot.columns[group5_columns],
             labels={"variable": ""}
                     #;"value":"WU [m^3/s]"}
                      )
fig5.update_layout(
    yaxis_title='WU [m^3/s]'
)

fig5.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Box(y=NLU_regions_plot.columns[group1_columns]))
#fig.add_trace(go.Box(y=y1))

#fig.add_trace(go.Box(y=NLU_regions_plot.columns[group1_columns], name='Regiones con más consumo',
#                marker_color = 'indianred'))

#fig = px.box(NLU_regions_plot, y=NLU_regions_plot.columns[group1_columns],
#             labels={"variable": ""}
#                     #;"value":"WU [m^3/s]"}
#                      )
#fig.update_layout(
#    yaxis_title='WU [m^3/s]'
#)

# # Obtener el año correspondiente al mínimo, máximo y outliers para cada columna del primer grupo
# for i, col in enumerate(NLU_regions_plot.columns[group1_columns]):
#     min_year = NLU_regions_plot.loc[NLU_regions_plot[col] == NLU_regions_plot[col].min(), 'Years'].values[0]
#     max_year = NLU_regions_plot.loc[NLU_regions_plot[col] == NLU_regions_plot[col].max(), 'Years'].values[0]
# #    outliers_years = NLU_regions_plot.loc[NLU_regions_plot[col].isin(fig1.data[i]['outliers']), 'Years'].values

#     # Agregar la anotación al lado de la figura de boxplot
#     fig.add_annotation(
#         x=1,  # Posición x de la anotación
#         y=fig.data[i]['q3'],  # Posición y de la anotación
#         #text=f"Min Year: {min_year}<br>Max Year: {max_year}<br>Outliers: {', '.join(outliers_years)}",
#         text=f"Min Year: {min_year}<br>Max Year: {max_year}",
#         showarrow=False,
#         xshift=10,  # Ajuste horizontal de la anotación
#         font=dict(size=10)
#     )


# Agregar anotaciones para cada figura de boxplot
for i, data in enumerate(fig.data):
    annotation_text = f"Año: {NLU_regions_plot.iloc[data.x]['Years']}"

    fig.add_annotation(
        x=data.x,
        y=data.y,
        text=annotation_text,
        showarrow=False,
        font=dict(size=10),
        xanchor='left',
        yanchor='middle'
    )

# Mostrar el gráfico del primer grupo con las anotaciones
fig1.show()

In [ ]:
#Graph of yearly regional (Non Land) Water Use [m3/s]

x_column = NLU_regions_plot.columns[0]
NLU_regions_melted = NLU_regions_plot.melt(id_vars=x_column)
fig = px.bar(NLU_regions_melted, x=x_column, y='value', color='variable',
             barmode='stack', title = "Non Land Water Use in Chile",labels={'Years':'Years', 'value':'Water use [m³/s]'}) 
fig.update_layout(legend_title_text="Regions")

fig.show()

fig2 = px.bar(NLU_regions_melted, x=x_column, y='value', color='variable',
             barmode='stack', title="Uso de agua no relacionado con la tierra en CHile", labels={'Years':'Años', 'value':'Uso de agua [m³/s]'}) 
fig2.update_layout(legend_title_text="Regiones")

fig2.show()

### Regional Water Use Analysis

In [ ]:
#Last 10 years WU
NLU_regions_plot2 = NLU_regions_plot.loc[NLU_regions_plot['Years'] > 2010]

In [ ]:
#Regional latitude order
ord_col = ['Years','REGIÓN DE ARICA Y PARINACOTA','REGIÓN DE TARAPACÁ', 'REGIÓN DE ANTOFAGASTA',
       'REGIÓN DE ATACAMA', 'REGIÓN DE COQUIMBO', 'REGIÓN DE VALPARAÍSO','REGIÓN METROPOLITANA DE SANTIAGO',
       "REGIÓN DEL LIBERTADOR GENERAL BERNARDO O'HIGGINS", 'REGIÓN DEL MAULE','REGIÓN DE ÑUBLE',
       'REGIÓN DEL BIOBÍO', 'REGIÓN DE LA ARAUCANÍA', 'REGIÓN DE LOS RÍOS', 'REGIÓN DE LOS LAGOS',
       'REGIÓN DE AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO',
       'REGIÓN DE MAGALLANES Y DE LA ANTÁRTICA CHILENA']

NLU_regions_plot2 = NLU_regions_plot2.reindex(columns=ord_col)

mean_values = NLU_regions_plot2.iloc[:, 1:].shift().mean()
#mean_row = pd.DataFrame([mean_values], columns=NLU_regions_plot2.columns[1:])

fig = px.bar(mean_values, x=mean_values.index, y=mean_values.values, 
             title='Yearly mean water use during the last 10 years by region ordered from north to south',
            labels={'index':'','y':'Water use [m³/s]'})
fig.update_xaxes(tickangle=60, tickfont=dict(size=9))
fig.update_layout(width = 1000, height = 800)
fig.show()

In [ ]:
NLU_regions_plot2.head()

In [ ]:
#x_ine = [1940, 1952, 1960, 1970, 1982, 1992, 2002, 2017]
#y_ine = [5023539, 5932995, 7374115, 8884768, 11329736, 13348401, 15116435, 17574003]
#dict_ine = dict(zip(x_ine, y_ine))
#print(dict_ine)

In [ ]:
#y_interp = scipy.interpolate.interp1d(x_ine, y_ine)

##find y-value associated with x-value of 13
#print(y_interp(2014))
#print(y_interp(2015))
#print(y_interp(2017))



In [ ]:
#Population = []
#pop = pd.DataFrame(index=year_list, columns=Population)
#for year in pop[]:
##    while year not in x_ine:

#for y in year_list:
#    while y not in x_ine:
        
#        valor = input(f"Ingrese el valor para [{indice}, {columna}]: ")
#        df.loc[indice, columna] = valor

#print(df)

### Type of Use Anlysis

In [ ]:
#r'/mnt/home2/mari/Tesis/NLU/csv/Total/DrinkingWater
#r'/mnt/home2/mari/Tesis/NLU/csv/Total/Energy
#r'/mnt/home2/mari/Tesis/NLU/csv/Total/Industry
#r'/mnt/home2/mari/Tesis/NLU/csv/Total/Livestock
#r'/mnt/home2/mari/Tesis/NLU/csv/Total/Mining
#r'/mnt/home2/mari/Tesis/NLU/csv/Total/WaterBodies

In [ ]:
DW_nation = pd.read_csv(r'/mnt/home2/mari/Tesis/NLU/csv/Total/DrinkingWater/CR2WU_NLU_DrinkingWater_historical_1950-2020_nation.csv')
DW_regions = pd.read_csv(r'/mnt/home2/mari/Tesis/NLU/csv/Total/DrinkingWater/CR2WU_NLU_DrinkingWater_historical_1950-2020_regions.csv')

In [ ]:
#Total use = DW+Energy+....
Total_nation = pd.read_csv(r'/mnt/home2/mari/Tesis/NLU/csv/Total/CR2WU_NLU_Total_historical_1950-2020_nation.csv')
Energy_nation = pd.read_csv(r'/mnt/home2/mari/Tesis/NLU/csv/Total/Energy/CR2WU_NLU_Energy_historical_1950-2020_nation.csv')
Industry_nation = pd.read_csv(r'/mnt/home2/mari/Tesis/NLU/csv/Total/Industry/CR2WU_NLU_Industry_historical_1950-2020_nation.csv')
Livestock_nation = pd.read_csv(r'/mnt/home2/mari/Tesis/NLU/csv/Total/Livestock/CR2WU_NLU_Livestock_historical_1950-2020_nation.csv')
Mining_nation = pd.read_csv(r'/mnt/home2/mari/Tesis/NLU/csv/Total/Mining/CR2WU_NLU_Mining_historical_1950-2020_nation.csv')
WaterBodies_nation = pd.read_csv(r'/mnt/home2/mari/Tesis/NLU/csv/Total/WaterBodies/CR2WU_NLU_WaterBodies_historical_1950-2020_nation.csv')

In [ ]:
#suma sin purificar 
E_tot = Energy_nation['1'].sum()
I_tot = Industry_nation['1'].sum()
L_tot = Livestock_nation['1'].sum()
M_tot = Mining_nation['1'].sum()
W_tot = WaterBodies_nation['1'].sum()
D_tot = DW_nation['1'].sum()
Total_tot = Total_nation['1'].sum()

In [ ]:
sum_tot = D_tot+E_tot+I_tot+L_tot+M_tot+W_tot
Total_tot-sum_tot #-1.4551915228366852e-11 diría que es error de truncamiento

In [ ]:
#Database cleaning
DW_nation['Date'] = pd.to_datetime(DW_nation["Date"])
DW_nation['Date'] = DW_nation['Date'].dt.strftime('%Y')
DW_nation = DW_nation.rename(columns={"Date": "Years","1":"NLU"})
DW_nation['Years'] = DW_nation['Years'].astype(int)

DW_regions['Date'] = pd.to_datetime(DW_regions["Date"])
DW_regions['Date'] = DW_regions['Date'].dt.strftime('%Y')
DW_regions = DW_regions.rename(columns={"Date": "Years"})
DW_regions['Years'] = DW_regions['Years'].astype(int)

#Adding Region's names
DW_regions_plot = DW_regions.rename(columns=cod_reg_nom_reg)

#NLU national 2 decimal
DW_nation_ = DW_nation
DW_nation_['NLU'] = DW_nation_['NLU'].round(decimals=2)

In [ ]:
DW_pop = DW_nation_[DW_nation_['Years'] >= 2002].reset_index(drop=True) #using NLU_nation_ 2 decimals NLU
#NLU_pop['Population']=pop_years_g['Population'] #usar este cuando corrija lo de arriba (crear pop years grouped)
DW_pop['Population']=pop_years['Population']
DW_pop.head()

In [ ]:
fig = px.scatter(DW_pop, x="Population", y="NLU", 
              title="Population and Drinking Water Use in Chile", 
              labels={'Population':'Population', 'NLU':'Drinking water use [m³/s]'}) 

fig.show()
#fig.write_image(r"/mnt/home2/mari/Tesis/pop_dw.png")

In [ ]:
DW_pop['Log_Pop']=np.log(DW_pop['Population'])
DW_pop['Log_NLU_DW']=np.log(DW_pop['NLU'])
fig = px.scatter(DW_pop, x="Log_Pop", y="Log_NLU_DW", 
              title="Population and Drinking Water Use in Chile", 
              labels={'Log_Pop':'Log Population', 'NLU':'Log Drinking Water use'}) 

fig.show()
#fig.write_image(r"/mnt/home2/mari/Tesis/logpopul_logdw.png")

In [ ]:
DW_pop.head()

In [ ]:
fig = px.scatter(DW_pop, x="Years", y="NLU/Pop", trendline="ols",
              title="Population and Drinking Water Use in Chile", 
              labels={'Years':'Years', 'NLU/Pop':'Drinking Water use per cápita [m³/s pp]'})
fig.update_layout(yaxis_tickformat = '.1e')


fig.show()

In [ ]:
DW_pop['NLU/Pop']=DW_pop['NLU']/DW_pop['Population']
DW_pop['Log_NLU/Pop' ]=np.log(DW_pop['NLU/Pop'])
fig = px.scatter(DW_pop, x="Log_Pop", y="Log_NLU/pop", 
              title="Population and Drinking Water Use in Chile", 
              labels={'Log_Pop':'Log Population', 'Log_NLU/pop':'Log Drinking Water use per cápita'})

fig.show()